<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Audio_PostProcessor_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

View this notebook on GitHub: <a href="https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Audio_PostProcessor_Colab.ipynb" target="_parent">Skquark/AEI-Colab-Notebooks/Audio_PostProcessor_Colab.ipynb</a>



# Audio Post-Processor — trim, normalize, denoise, convert for TTS / VC / podcast output

A companion to all our TTS, voice-cloning, and voice-conversion notebooks. Takes the **raw audio output** from Qwen3-TTS, VoxCPM2, Higgs, MisoTTS, MOSS, dots.tts, Fish, Kokoro, OpenVoice V2, or any other pipeline, and turns it into a **clean, ready-to-publish** audio file: trimmed, denoised, loudness-normalized (LUFS), and in the format you need (WAV / MP3 / FLAC / OGG / Opus / M4A / AAC / AIFF).

**Stack**: [`pydub`](https://github.com/jiaaro/pydub) (MIT) for format conversion + silence detect · [`imageio-ffmpeg`](https://github.com/imageio/imageio-ffmpeg) (BSD-4) for the static ffmpeg binary · [`soundfile`](https://github.com/bastibe/python-soundfile) (BSD-3, libsndfile) for fast WAV/FLAC/OGG/AIFF round-trips · [`librosa`](https://github.com/librosa/librosa) (ISC) for DSP analysis · [`pyloudnorm`](https://github.com/csteinmetz1/pyloudnorm) (MIT) for ITU-R BS.1770-4 LUFS / LKFS broadcast-standard normalization · [`noisereduce`](https://github.com/timsainb/noisereduce) (MIT) for spectral-gating noise reduction. **All CPU-only. No GPU required.**

Eight pipeline stages (all optional, all tunable):
1. **Load** — auto-detect format from extension. 8 input formats.
2. **High-pass filter** — cut rumble & mic handling noise (configurable cutoff, default 0=off).
3. **Denoise** — spectral gating via noisereduce. Tunable strength.
4. **LUFS normalize** — broadcast-standard ITU-R BS.1770-4 loudness target (default -16 LUFS for podcast).
5. **Peak limit** — guard against clipping (default -1 dBFS).
6. **Trim** — keep only the time range you need.
7. **Split** — split by silence, or into N-second chunks.
8. **Export** — 8 output formats with configurable bitrate & sample rate.

Plus 8 export formats: `.wav` (lossless), `.flac` (lossless compressed), `.aiff` (lossless), `.mp3` (universal), `.ogg` (open-source), `.opus` (streaming), `.m4a` (Apple), `.aac` (raw ADTS).



In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs pydub, imageio-ffmpeg, soundfile, librosa, pyloudnorm, noisereduce, scipy, pedalboard, resemble-enhance.

import os, sys, subprocess
# Gradio pinned to a specific version for reproducibility. tqdm is required for
# Resemble Enhance's download.py (uses torch.hub.download_url_to_file without
# its own progress bar). pedalboard uses pybind11/numpy internally.
pkgs = ['pydub', 'imageio-ffmpeg', 'soundfile', 'librosa', 'pyloudnorm', 'noisereduce', 'scipy',
        'pedalboard', 'resemble-enhance', 'tqdm==4.66.5', 'gradio==5.33.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs, check=True)
# librosa + numba pin
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'numba==0.58.1', 'numpy<2.1'], check=False)

import soundfile as sf
import pydub
import imageio_ffmpeg
import librosa, pyloudnorm, noisereduce, scipy
import pedalboard
try:
    import resemble_enhance
    _have_resemble = True
    _re_ver = getattr(resemble_enhance, '__version__', 'unknown')
except Exception as _e:
    _have_resemble = False
    _re_ver = f'not installed ({_e})'
import numpy as np
print('[Step 1] Installed:')
print(f'  pydub          {pydub.__version__}')
print(f'  soundfile      {sf.__version__} (libsndfile backend)')
print(f'  imageio-ffmpeg {imageio_ffmpeg.__version__} (ffmpeg at {imageio_ffmpeg.get_ffmpeg_exe()})')
print(f'  librosa        {librosa.__version__}')
print(f'  pyloudnorm     {pyloudnorm.__version__}')
print(f'  noisereduce    {noisereduce.__version__}')
print(f'  scipy          {scipy.__version__}')
print(f'  pedalboard     {pedalboard.__version__} (Spotify Audio Intelligence Lab, GPL-3.0)')
print(f'  resemble-enhance  {_re_ver} (Resemble AI, MIT)' + ('  — AI Enhance tab disabled' if not _have_resemble else ''))
print(f'  numpy          {np.__version__}')
print('[Step 1] Done.')


In [ ]:
# @title STEP 2 — Pre-cache a small demo audio
# @markdown Writes a short test sine sweep + spoken phrase to /content/audio_examples/ for the Examples buttons.

import os, urllib.request, subprocess
DEST = '/content/audio_examples'
os.makedirs(DEST, exist_ok=True)

# Try to download a public-domain sample; fall back to generated tone
SAMPLE = os.path.join(DEST, 'sample_voice_48k.wav')
URL = 'https://github.com/jiaaro/pydub/raw/master/test/data/test1.wav'
try:
    urllib.request.urlretrieve(URL, SAMPLE)
    print(f'[Step 2] Downloaded: {SAMPLE} ({os.path.getsize(SAMPLE)/1024:.1f} KB)')
except Exception as e:
    # Synthesize a 3-second tone as fallback
    import numpy as np
    import soundfile as sf
    sr = 48000
    t = np.linspace(0, 3, sr * 3, endpoint=False)
    sig = 0.2 * np.sin(2 * np.pi * 440 * t)
    sf.write(SAMPLE, sig, sr)
    print(f'[Step 2] Generated fallback tone: {SAMPLE}')

# Generate a "noise" example for the Denoise tab to chew on
NOISY = os.path.join(DEST, 'sample_noisy_48k.wav')
import numpy as np, soundfile as sf
sr = 48000
t = np.linspace(0, 3, sr * 3, endpoint=False)
tone = 0.2 * np.sin(2 * np.pi * 440 * t)
noise = 0.05 * np.random.randn(sr * 3)
sf.write(NOISY, tone + noise, sr)
print(f'[Step 2] Generated noisy sample: {NOISY}')

print('[Step 2] Done.')



In [ ]:
# @title STEP 3 — Core pipeline (lazy imports, presets)
# @markdown All processing functions + preset dictionary. Imported by the Gradio UI in Step 4.

import os, sys, time, io, json, zipfile
import numpy as np
from scipy import signal as scipy_signal

# Lazy imports to keep startup fast
_pd = None
_sf = None
_nr = None
_ln = None
_lr = None
_ffmpeg = None
_pb = None
_re = None
_re_device = None

def _ensure():
    global _pd, _sf, _nr, _ln, _lr, _ffmpeg, _pb, _re, _re_device
    if _pd is None:
        import pydub
        import soundfile
        import noisereduce
        import pyloudnorm
        import librosa
        import imageio_ffmpeg
        import pedalboard
        from pedalboard import Pedalboard, Gain, Compressor, Limiter, HighpassFilter, LowpassFilter, LadderFilter, Reverb, Delay, Chorus, Phaser, Distortion, Clipping, PitchShift, Bitcrush, Resample, MP3Compressor, GSMFullRateCompressor, Convolution
        _pd = pydub
        _sf = soundfile
        _nr = noisereduce
        _ln = pyloudnorm
        _lr = librosa
        _ffmpeg = imageio_ffmpeg
        _pb = {
            'Pedalboard': Pedalboard,
            'Gain': Gain,
            'Compressor': Compressor,
            'Limiter': Limiter,
            'HighpassFilter': HighpassFilter,
            'LowpassFilter': LowpassFilter,
            'LadderFilter': LadderFilter,
            'Reverb': Reverb,
            'Delay': Delay,
            'Chorus': Chorus,
            'Phaser': Phaser,
            'Distortion': Distortion,
            'Clipping': Clipping,
            'PitchShift': PitchShift,
            'Bitcrush': Bitcrush,
            'Resample': Resample,
            'MP3Compressor': MP3Compressor,
            'GSMFullRateCompressor': GSMFullRateCompressor,
            'Convolution': Convolution,
        }
        try:
            from resemble_enhance.enhancer.inference import denoise as _re_denoise_fn
            from resemble_enhance.enhancer.inference import enhance as _re_enhance_fn
            import torch as _torch
            _re_device = 'cuda' if _torch.cuda.is_available() else 'cpu'
            _re = {'denoise': _re_denoise_fn, 'enhance': _re_enhance_fn, 'torch': _torch}
        except Exception:
            _re = None
            _re_device = 'cpu'
    return _pd, _sf, _nr, _ln, _lr, _ffmpeg, _pb, _re

# ── 4 quick-process presets (mirrors README + UI dropdown) ────
PRESETS = [
    {
        "id": "podcast",
        "name": "Podcast (warm, normalized)",
        "desc": "High-pass filter (80 Hz) + LUFS normalize to -16 LUFS + light denoise. Two-speaker friendly.",
        "pipeline": {
            "highpass_hz": 80,
            "denoise": true,
            "denoise_strength": 0.5,
            "target_lufs": -16.0,
            "peak_db": -1.0
        }
    },
    {
        "id": "music",
        "name": "Music (preserves dynamics)",
        "desc": "LUFS normalize to -14 LUFS (Spotify target) + light denoise. No high-pass, preserves bass.",
        "pipeline": {
            "highpass_hz": 0,
            "denoise": true,
            "denoise_strength": 0.3,
            "target_lufs": -14.0,
            "peak_db": -1.0
        }
    },
    {
        "id": "speech",
        "name": "Speech (broadcast-ready)",
        "desc": "High-pass 100 Hz + strong denoise + LUFS -23 (EBU R128 broadcast) + peak limit -2 dBFS.",
        "pipeline": {
            "highpass_hz": 100,
            "denoise": true,
            "denoise_strength": 0.7,
            "target_lufs": -23.0,
            "peak_db": -2.0
        }
    },
    {
        "id": "broadcast",
        "name": "Broadcast (EBU R128 strict)",
        "desc": "LUFS -23 + true-peak limit -1 dBTP. Conforms to EBU R128 / ATSC A/85 broadcast specs.",
        "pipeline": {
            "highpass_hz": 0,
            "denoise": true,
            "denoise_strength": 0.4,
            "target_lufs": -23.0,
            "peak_db": -1.0
        }
    },
    {
        "id": "studio_polish",
        "name": "Studio Polish (AI bandwidth extension)",
        "desc": "Resemble Enhance (44.1 kHz AI model: denoise + restore high-freq + reduce artifacts) + LUFS -16 + peak -1. Best for thin / compressed TTS or noisy speech. Slow (10-60s on CPU, 1-5s on GPU).",
        "pipeline": {
            "highpass_hz": 80,
            "resemble_enhance": true,
            "re_solver": "Midpoint",
            "re_nfe": 64,
            "re_tau": 0.5,
            "re_lambd": 0.5,
            "re_denoise_first": false,
            "denoise": false,
            "target_lufs": -16.0,
            "peak_db": -1.0
        }
    }
]

# ── 8 supported formats (extension + codec hint) ──────────────
IO_FORMATS = ["wav", "mp3", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
CODEC_HINTS = {"wav": ["pcm_s16le", "WAV (16-bit PCM)"], "mp3": ["libmp3lame", "MP3 (LAME)"], "flac": ["flac", "FLAC (lossless)"], "ogg": ["libvorbis", "OGG Vorbis"], "opus": ["libopus", "Opus (streaming-optimized)"], "m4a": ["aac", "M4A / AAC (Apple-friendly)"], "aac": ["aac", "AAC ADTS"], "aiff": ["pcm_s16be", "AIFF (16-bit PCM)"]}


def get_preset(preset_id):
    for p in PRESETS:
        if p['id'] == preset_id:
            return dict(p['pipeline'])
    raise ValueError(f'Unknown preset: {preset_id!r}')


def _ext_to_format(filename: str) -> str:
    ext = os.path.splitext(filename)[1].lower().lstrip('.')
    return ext if ext in IO_FORMATS else 'wav'


def _load_audio(input_path: str):
    """Load any supported audio file as (samples float32 in [-1, 1], sample_rate int)."""
    pd, sf, _, _, _, _, _pb, _re = _ensure()
    ext = _ext_to_format(input_path)
    if ext in ('wav', 'flac', 'ogg', 'aiff'):
        # Fast libsndfile path
        data, sr = sf.read(input_path, dtype='float32', always_2d=False)
        if data.ndim > 1:
            data = data.mean(axis=1)
        return data, sr
    # pydub path for compressed formats
    audio = pd.AudioSegment.from_file(input_path)
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / (2 ** (8 * audio.sample_width - 1))
    if audio.channels > 1:
        samples = samples.reshape(-1, audio.channels).mean(axis=1)
    return samples, audio.frame_rate


def _save_audio(samples: np.ndarray, sr: int, output_path: str, format_ext: str = None,
                bitrate: str = '192k', normalize_to_dbfs: float = None):
    """Save samples to a file in any supported format. Optionally apply peak normalize just before export."""
    pd, sf, _, _, _, ffmpeg_pkg, _pb, _re = _ensure()
    if format_ext is None:
        format_ext = _ext_to_format(output_path)
    if format_ext not in IO_FORMATS:
        raise ValueError(f'Unsupported output format: {format_ext!r}')
    # Optional peak normalize
    if normalize_to_dbfs is not None and len(samples) > 0:
        peak = float(np.max(np.abs(samples)))
        if peak > 0:
            target_linear = 10 ** (normalize_to_dbfs / 20.0)
            samples = samples * (target_linear / peak)
    # pydub path for everything
    int_samples = np.clip(samples, -1.0, 1.0)
    int_samples = (int_samples * 32767).astype(np.int16).tobytes()
    audio = pd.AudioSegment(
        data=int_samples,
        sample_width=2,
        frame_rate=sr,
        channels=1,
    )
    if format_ext in ('wav', 'aiff'):
        # Use soundfile for lossless (faster, no ffmpeg)
        sf.write(output_path, samples, sr, subtype='PCM_16')
        return output_path
    codec, _ = CODEC_HINTS[format_ext]
    out_params = ['-acodec', codec]
    if format_ext in ('mp3', 'ogg', 'opus', 'm4a', 'aac'):
        out_params += ['-b:a', bitrate]
    audio.export(output_path, format=format_ext, parameters=out_params)
    return output_path


def _highpass(samples: np.ndarray, sr: int, cutoff_hz: float) -> np.ndarray:
    if cutoff_hz <= 0:
        return samples
    b, a = scipy_signal.butter(4, cutoff_hz / (sr / 2), btype='highpass')
    return scipy_signal.filtfilt(b, a, samples).astype(np.float32)


def _denoise(samples: np.ndarray, sr: int, strength: float) -> np.ndarray:
    if strength <= 0:
        return samples
    _, _, nr, _, _, _, _pb, _re = _ensure()
    return nr.reduce_noise(y=samples, sr=sr, prop_decrease=min(max(strength, 0.0), 1.0)).astype(np.float32)


def _measure_lufs(samples: np.ndarray, sr: int) -> float:
    try:
        _, _, _, ln, _, _, _pb, _re = _ensure()
        meter = ln.Meter(sr)
        return float(meter.integrated_loudness(samples))
    except Exception:
        return float('nan')


def _lufs_normalize(samples: np.ndarray, sr: int, target_lufs: float) -> np.ndarray:
    if target_lufs is None or target_lufs == 0:
        return samples
    _, _, _, ln, _, _, _pb, _re = _ensure()
    meter = ln.Meter(sr)
    try:
        loudness = meter.integrated_loudness(samples)
        if not np.isfinite(loudness):
            return samples
        return ln.normalize.loudness(samples, loudness, target_lufs).astype(np.float32)
    except Exception:
        return samples


def _analyze(samples: np.ndarray, sr: int) -> dict:
    """Return a dict of stats for the Compare tab + log output."""
    if len(samples) == 0:
        return {'duration_s': 0.0, 'peak_dbfs': float('nan'), 'rms_dbfs': float('nan'),
                'lufs': float('nan'), 'sample_rate': sr, 'samples': 0}
    peak = float(np.max(np.abs(samples)))
    rms = float(np.sqrt(np.mean(samples ** 2)))
    peak_dbfs = 20.0 * np.log10(max(peak, 1e-9))
    rms_dbfs = 20.0 * np.log10(max(rms, 1e-9))
    lufs = _measure_lufs(samples, sr)
    return {
        'duration_s': float(len(samples) / sr),
        'peak_dbfs': peak_dbfs,
        'rms_dbfs': rms_dbfs,
        'lufs': lufs,
        'sample_rate': sr,
        'samples': int(len(samples)),
    }


# ── 4 effects-chain presets (used by Effects Chain tab) ──────
EFFECT_CHAIN_PRESETS = {
    'podcast_voice': {
        'name': 'Podcast Voice (warm, broadcast-ready)',
        'chain': [
            ('HighpassFilter', {'cutoff_hz': 80}),
            ('Compressor', {'threshold_db': -20, 'ratio': 3.0, 'attack_ms': 5, 'release_ms': 100}),
            ('Gain', {'gain_db': 3.0}),
            ('Limiter', {'threshold_db': -1.0, 'release_ms': 100}),
        ],
    },
    'vocal_cleaning': {
        'name': 'Vocal Cleaning (denoise + tighten)',
        'chain': [
            ('HighpassFilter', {'cutoff_hz': 100}),
            ('Compressor', {'threshold_db': -22, 'ratio': 4.0, 'attack_ms': 10, 'release_ms': 120}),
            ('Limiter', {'threshold_db': -1.0, 'release_ms': 80}),
        ],
    },
    'music_mastering': {
        'name': 'Music Mastering (loud + clean)',
        'chain': [
            ('Compressor', {'threshold_db': -15, 'ratio': 2.5, 'attack_ms': 20, 'release_ms': 150}),
            ('Gain', {'gain_db': 2.0}),
            ('Limiter', {'threshold_db': -0.5, 'release_ms': 50}),
        ],
    },
    'lofi_tape': {
        'name': 'Lo-Fi Tape (degraded + warm)',
        'chain': [
            ('HighpassFilter', {'cutoff_hz': 100}),
            ('LowpassFilter', {'cutoff_hz': 8000}),
            ('Bitcrush', {'bit_depth': 10}),
            ('Chorus', {'rate_hz': 0.3, 'depth': 0.4, 'centre_delay_ms': 12.0, 'feedback': 0.0, 'mix': 0.4}),
            ('Reverb', {'room_size': 0.6, 'damping': 0.4, 'wet_level': 0.3, 'dry_level': 0.7, 'width': 1.0}),
        ],
    },
}


def apply_effects_chain(samples: np.ndarray, sr: int, chain: list) -> tuple:
    """Apply a list of (effect_name, kwargs) tuples via pedalboard.
    Returns (effected_samples, applied_log)."""
    _, _, _, _, _, _, pb, _re = _ensure()
    Pedalboard = pb['Pedalboard']
    plugins = []
    log = []
    for effect_name, params in chain:
        if effect_name not in pb:
            log.append(f'  [skip] Unknown effect: {effect_name}')
            continue
        try:
            plugin = pb[effect_name](**params)
            plugins.append(plugin)
            log.append(f'  [+] {effect_name}({params})')
        except Exception as e:
            log.append(f'  [skip] {effect_name}: {e}')
    if not plugins:
        return samples, chr(10).join(log)
    board = Pedalboard(plugins)
    effected = board(samples, sr).astype(np.float32)
    return effected, chr(10).join(log)


def apply_chain_preset(preset_id: str, samples: np.ndarray, sr: int) -> tuple:
    """Apply a named chain preset. Returns (effected, log)."""
    if preset_id not in EFFECT_CHAIN_PRESETS:
        return samples, f'Unknown chain preset: {preset_id!r}'
    p = EFFECT_CHAIN_PRESETS[preset_id]
    log = [f'Chain preset: {p["name"]}']
    effected, apply_log = apply_effects_chain(samples, sr, p['chain'])
    log.append(apply_log)
    return effected, chr(10).join(log)


# ── Resemble Enhance wrappers ─────────────────────────────────
def res_enhance(samples: np.ndarray, sr: int, solver: str = 'Midpoint', nfe: int = 64,
                tau: float = 0.5, lambd: float = 0.1, denoise_first: bool = False,
                run_dir: str = None, device: str = None) -> dict:
    """AI speech enhancement via Resemble Enhance (denoise + bandwidth extension).
    Returns dict {wav (np.ndarray), sr (int), log (str), denoised (np.ndarray|None), denoised_sr (int|None)}.
    Auto-resamples input to 44.1 kHz mono (the model's native rate). lambd>0 enables the conditional
    denoiser path inside the enhancer. If denoise_first is True, runs the standalone denoiser first
    and feeds its output into the enhancer (highest-quality path, ~2x slower).
    run_dir (str|None): optional path to a custom model checkpoint directory. None = use the default
    (~/.cache/huggingface/hub via resemble_enhance.enhancer.download). device (str|None): optional
    'cuda' / 'cpu' override. None = use auto-detected device.
    """
    _, _, _, _, _, _, _, re = _ensure()
    if re is None:
        raise RuntimeError('resemble-enhance is not installed. Run Step 1 to install it.')
    if device is None:
        device = _re_device
    import torch as _torch
    wav = _torch.from_numpy(samples.astype(np.float32)).to(device)
    log = [f'Resemble Enhance: solver={solver}, nfe={nfe}, tau={tau}, lambd={lambd}, denoise_first={denoise_first}, device={device}, run_dir={run_dir or "<default>"}']
    denoised_out = None
    denoised_sr_out = None
    if denoise_first:
        dw, new_sr = re['denoise'](wav, sr, device, run_dir=run_dir)
        denoised_out = dw.cpu().numpy()
        denoised_sr_out = new_sr
        log.append(f'  Denoised: {dw.shape[-1]/new_sr:.2f}s @ {new_sr} Hz')
        wav = dw
        sr = new_sr
    ew, new_sr = re['enhance'](wav, sr, device, nfe=int(nfe), solver=str(solver).lower(),
                                lambd=float(lambd), tau=float(tau), run_dir=run_dir)
    out = ew.cpu().numpy().astype(np.float32)
    log.append(f'  Enhanced: {out.shape[-1]/new_sr:.2f}s @ {new_sr} Hz  ({device})')
    return {'wav': out, 'sr': new_sr, 'log': chr(10).join(log),
            'denoised': denoised_out, 'denoised_sr': denoised_sr_out}


def res_denoise(samples: np.ndarray, sr: int, run_dir: str = None, device: str = None) -> dict:
    """Standalone denoiser from Resemble Enhance (no bandwidth extension). Returns dict {wav, sr, log}."""
    _, _, _, _, _, _, _, re = _ensure()
    if re is None:
        raise RuntimeError('resemble-enhance is not installed. Run Step 1 to install it.')
    if device is None:
        device = _re_device
    import torch as _torch
    wav = _torch.from_numpy(samples.astype(np.float32)).to(device)
    log = [f'Resemble Denoiser (standalone): device={device}, run_dir={run_dir or "<default>"}']
    dw, new_sr = re['denoise'](wav, sr, device, run_dir=run_dir)
    out = dw.cpu().numpy().astype(np.float32)
    log.append(f'  Denoised: {out.shape[-1]/new_sr:.2f}s @ {new_sr} Hz  ({device})')
    return {'wav': out, 'sr': new_sr, 'log': chr(10).join(log)}


def batch_res_enhance(input_dir: str, output_dir: str, pattern: str = '*.wav',
                      recursive: bool = False, output_format: str = 'wav',
                      bitrate_kbps: int = 192, run_dir: str = None,
                      solver: str = 'Midpoint', nfe: int = 64, tau: float = 0.5,
                      lambd: float = 0.5, denoise_first: bool = False,
                      device: str = None, progress=None) -> dict:
    """Enhance every audio file in a directory via Resemble Enhance.
    Returns a summary dict {processed, failed, total, output_dir, log (str), elapsed_s (float)}.
    Per-file failures are logged and skipped; one bad file does not abort the batch.
    progress (callable|None): optional progress callback f(fraction, desc) -> None. Pass gr.Progress() from a Gradio handler."""
    import glob as _glob
    t0 = time.time()
    os.makedirs(output_dir, exist_ok=True)
    # Collect files
    if recursive:
        candidates = []
        for root, _, files in os.walk(input_dir):
            for f in files:
                candidates.append(os.path.join(root, f))
        files = sorted([p for p in candidates if _glob.fnmatch.fnmatch(os.path.basename(p).lower(), pattern.lower())])
    else:
        files = sorted(_glob.glob(os.path.join(input_dir, pattern)))
        # Also case-insensitive fallback
        if not files:
            files = sorted([os.path.join(input_dir, f) for f in os.listdir(input_dir)
                            if _glob.fnmatch.fnmatch(f.lower(), pattern.lower())])
    log = [f'Batch Resemble Enhance: {len(files)} file(s) matched "{pattern}" in {input_dir}']
    log.append(f'  Output dir: {output_dir}')
    log.append(f'  Format: {output_format}  Bitrate: {bitrate_kbps} kbps  Recursive: {recursive}')
    log.append(f'  Solver: {solver}  NFE: {nfe}  tau: {tau}  lambd: {lambd}  denoise_first: {denoise_first}')
    log.append('')
    processed = 0
    failed = 0
    for i, in_path in enumerate(files):
        rel = os.path.relpath(in_path, input_dir)
        base = os.path.splitext(os.path.basename(in_path))[0]
        # Preserve subdirectory structure if recursive
        if recursive and os.path.dirname(rel):
            sub_out = os.path.join(output_dir, os.path.dirname(rel))
            os.makedirs(sub_out, exist_ok=True)
            out_path = os.path.join(sub_out, f'{base}_enhanced.{output_format}')
        else:
            out_path = os.path.join(output_dir, f'{base}_enhanced.{output_format}')
        try:
            if progress is not None:
                progress((i + 0.1) / max(1, len(files)), desc=f'[{i+1}/{len(files)}] {os.path.basename(in_path)}')
            samples, sr = _load_audio(in_path)
            result = res_enhance(samples, sr, solver=solver, nfe=nfe, tau=tau, lambd=lambd,
                                 denoise_first=denoise_first, run_dir=run_dir, device=device)
            out_wav = result['wav']; new_sr = result['sr']
            br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
            _save_audio(out_wav, new_sr, out_path, format_ext=output_format, bitrate=br)
            processed += 1
            log.append(f'  [OK {i+1}/{len(files)}] {rel}  ->  {os.path.basename(out_path)}  ({out_wav.shape[-1]/new_sr:.2f}s)')
        except Exception as e:
            failed += 1
            log.append(f'  [FAIL {i+1}/{len(files)}] {rel}: {e}')
    if progress is not None:
        progress(1.0, desc='Done')
    elapsed = time.time() - t0
    log.append('')
    log.append(f'Finished: {processed} ok, {failed} failed, {elapsed:.1f}s elapsed')
    return {
        'processed': processed,
        'failed': failed,
        'total': len(files),
        'output_dir': output_dir,
        'elapsed_s': elapsed,
        'log': chr(10).join(log),
    }

def process_audio(input_path: str, output_path: str, pipeline: dict, bitrate: str = None) -> dict:
    """Apply a pipeline dict (matching the preset schema) to an input file.
    Returns a dict with 'loaded' and 'final' stats blocks.
    bitrate (str|None): optional bitrate string (e.g. '192k') for compressed output formats. None = use _save_audio default ('192k')."""
    samples, sr = _load_audio(input_path)
    loaded = _analyze(samples, sr)
    log = []
    log.append(f'Loaded: {os.path.basename(input_path)}')
    log.append(f'  Duration: {loaded["duration_s"]:.2f}s  SR: {loaded["sample_rate"]} Hz  Channels: 1 (mono)')
    log.append(f'  Peak: {loaded["peak_dbfs"]:.1f} dBFS  RMS: {loaded["rms_dbfs"]:.1f} dBFS  LUFS: {loaded["lufs"]:.1f}')
    log.append('')

    if pipeline.get('highpass_hz', 0) > 0:
        samples = _highpass(samples, sr, float(pipeline['highpass_hz']))
        log.append(f'[1] High-pass filter @ {float(pipeline["highpass_hz"]):.0f} Hz')
    else:
        log.append('[1] High-pass: skipped')

    if pipeline.get('resemble_enhance', False):
        # AI enhancement step (denoise + bandwidth extension, 44.1 kHz model)
        _, _, _, _, _, _, _, re = _ensure()
        if re is None:
            log.append('[2] Resemble Enhance: SKIPPED (package not installed)')
        else:
            res_kwargs = {
                'solver': str(pipeline.get('re_solver', 'Midpoint')),
                'nfe': int(pipeline.get('re_nfe', 64)),
                'tau': float(pipeline.get('re_tau', 0.5)),
                'lambd': float(pipeline.get('re_lambd', 0.5)),
                'denoise_first': bool(pipeline.get('re_denoise_first', False)),
            }
            out = res_enhance(samples, sr, **res_kwargs)
            samples = out['wav']; sr = out['sr']
            log.append(f'[2] Resemble Enhance: solver={res_kwargs["solver"]} nfe={res_kwargs["nfe"]} tau={res_kwargs["tau"]:.2f} lambd={res_kwargs["lambd"]:.2f} denoise_first={res_kwargs["denoise_first"]}')
        if pipeline.get('denoise', False):
            s = float(pipeline.get('denoise_strength', 0.5))
            samples = _denoise(samples, sr, s)
            log.append(f'[3] Denoise (spectral gate, strength={s:.2f}) — post-AI polish')
        else:
            log.append('[3] Denoise: skipped')
    elif pipeline.get('denoise', False):
        s = float(pipeline.get('denoise_strength', 0.5))
        samples = _denoise(samples, sr, s)
        log.append(f'[2] Denoise (spectral gate, strength={s:.2f})')
    else:
        log.append('[2/3] Denoise: skipped')

    if pipeline.get('target_lufs', 0) != 0:
        t = float(pipeline['target_lufs'])
        samples = _lufs_normalize(samples, sr, t)
        log.append(f'[4] LUFS normalize to {t:.1f} LUFS (ITU-R BS.1770-4)')
    else:
        log.append('[4] LUFS normalize: skipped')

    peak_db = float(pipeline.get('peak_db', -1.0))
    samples = samples.astype(np.float32)
    final = _analyze(samples, sr)
    log.append(f'[5] Peak limit: {peak_db:.1f} dBFS (applied at export)')
    log.append('')
    log.append(f'Output:')
    log.append(f'  Duration: {final["duration_s"]:.2f}s  SR: {final["sample_rate"]} Hz')
    log.append(f'  Peak: {final["peak_dbfs"]:.1f} dBFS  RMS: {final["rms_dbfs"]:.1f} dBFS  LUFS: {final["lufs"]:.1f}')

    _save_audio(samples, sr, output_path, normalize_to_dbfs=peak_db, bitrate=bitrate or '192k')
    final['file_size_kb'] = os.path.getsize(output_path) / 1024
    log.append(f'  File: {output_path} ({final["file_size_kb"]:.1f} KB)')
    return {
        'loaded': loaded,
        'final': final,
        'log': chr(10).join(log),
        'samples': samples,
        'sample_rate': sr,
    }


def trim_audio(input_path: str, output_path: str, start_s: float, end_s: float) -> dict:
    samples, sr = _load_audio(input_path)
    loaded = _analyze(samples, sr)
    s = max(0, int(start_s * sr))
    e = min(len(samples), int(end_s * sr))
    out = samples[s:e]
    _save_audio(out, sr, output_path)
    final = _analyze(out, sr)
    return {
        'loaded': loaded, 'final': final,
        'samples': out, 'sample_rate': sr,
        'log': f'Trimmed: {start_s:.2f}s .. {end_s:.2f}s ({s} .. {e} samples)\nDuration: {loaded["duration_s"]:.2f}s -> {final["duration_s"]:.2f}s\nFile: {output_path}',
    }


def split_by_silence(input_path: str, output_dir: str, min_silence_ms: int = 500,
                      silence_thresh_dbfs: int = -40) -> list:
    pd, _, _, _, _, _, _pb, _re = _ensure()
    audio = pd.AudioSegment.from_file(input_path)
    chunks = pd.silence.split_on_silence(
        audio,
        min_silence_len=min_silence_ms,
        silence_thresh=silence_thresh_dbfs,
        keep_silence=200,
    )
    os.makedirs(output_dir, exist_ok=True)
    paths = []
    base = os.path.splitext(os.path.basename(input_path))[0]
    for i, chunk in enumerate(chunks):
        p = os.path.join(output_dir, f'{base}_chunk_{i+1:03d}.wav')
        chunk.export(p, format='wav')
        paths.append(p)
    return paths


def split_into_chunks(input_path: str, output_dir: str, chunk_s: float = 10.0) -> list:
    samples, sr = _load_audio(input_path)
    chunk_n = int(chunk_s * sr)
    os.makedirs(output_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(input_path))[0]
    paths = []
    for i in range(0, len(samples), chunk_n):
        chunk = samples[i:i + chunk_n]
        if len(chunk) < sr * 0.1:  # skip tiny trailing chunks < 100ms
            continue
        p = os.path.join(output_dir, f'{base}_chunk_{i//chunk_n + 1:03d}.wav')
        _save_audio(chunk, sr, p)
        paths.append(p)
    return paths


def detect_silence_ranges(input_path: str, min_silence_ms: int = 500,
                          silence_thresh_dbfs: int = -40) -> list:
    pd, _, _, _, _, _, _pb, _re = _ensure()
    audio = pd.AudioSegment.from_file(input_path)
    ranges = pd.silence.detect_silence(
        audio,
        min_silence_len=min_silence_ms,
        silence_thresh=silence_thresh_dbfs,
    )
    return [(s / 1000.0, e / 1000.0) for s, e in ranges]


def _make_waveform_png(samples: np.ndarray, sr: int, title: str = '') -> bytes:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 3), dpi=100)
    t = np.linspace(0, len(samples) / sr, num=min(len(samples), 50000), endpoint=False)
    s = samples[:len(t)]
    ax.plot(t, s, linewidth=0.5, color='#1976d2')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlim(0, t[-1] if len(t) > 0 else 1)
    ax.grid(True, alpha=0.3)
    if title:
        ax.set_title(title, fontsize=11)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100)
    plt.close(fig)
    return buf.getvalue()


print('[Step 3] Core pipeline ready.')



In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with ten tabs: **Quick Process** (5 presets incl. AI Studio Polish), **Trim & Split** (silence detect / chunk), **Normalize** (peak / RMS / LUFS), **Effects Chain** (pedalboard mastering), **Format Convert** (8 formats, bitrate, SR), **AI Enhance** (Resemble Enhance — denoise + bandwidth extension), **Denoise** (spectral gate), **Batch** (preset + directory), **Compare** (before/after waveform + stats), **Help** (when-to-use, format cheatsheet, citation).

import os, sys, time, json, io, zipfile
import numpy as np
import gradio as gr

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass

from IPython.display import clear_output, Audio, display, FileLink
clear_output()


# ── Materialize Gradio audio input (filepath string or dict) ──
def _materialize_audio_input(audio_obj) -> str:
    """Gradio can pass filepath string or a dict with 'name' key. Return a filepath."""
    if audio_obj is None:
        return None
    if isinstance(audio_obj, str):
        return audio_obj
    if isinstance(audio_obj, dict):
        return audio_obj.get('name') or audio_obj.get('path') or audio_obj.get('data')
    if isinstance(audio_obj, (list, tuple)) and len(audio_obj) >= 1:
        return _materialize_audio_input(audio_obj[0])
    return None


def _gr_audio(samples: np.ndarray, sr: int):
    """Return a Gradio audio value (filepath) for playback."""
    out = f'/content/audio_out/preview_{int(time.time()*1000)}.wav'
    os.makedirs(os.path.dirname(out), exist_ok=True)
    import soundfile as sf
    sf.write(out, samples, sr, subtype='PCM_16')
    return out


# ── Tab 1: Quick Process ───────────────────────────────────────
def ui_quick_process(audio_in, preset_id, output_format, bitrate_kbps,
                     progress=gr.Progress()):
    if audio_in is None:
        raise gr.Error('Upload an audio file first (WAV, MP3, FLAC, OGG, OPUS, M4A, AAC, AIFF).')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        pipeline = get_preset(preset_id)
        pipeline['export_format'] = output_format
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_{preset_id}.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else None
        t0 = time.time()
        result = process_audio(src, out_path, pipeline, bitrate=br)
        elapsed = time.time() - t0
        # Build preview from processed samples
        preview_path = _gr_audio(result['samples'], result['sample_rate'])
        log = result['log'] + f'\nTotal: {elapsed:.1f}s · bitrate: {br or "lossless"}'
        return preview_path, out_path, log
    except Exception as e:
        raise gr.Error(f'Processing failed: {e}')


# ── Tab 2: Trim & Split ───────────────────────────────────────
def ui_trim(audio_in, start_s, end_s, output_format):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_trimmed.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        result = trim_audio(src, out_path, float(start_s), float(end_s))
        preview_path = _gr_audio(result['samples'], result['sample_rate'])
        return preview_path, out_path, result['log']
    except Exception as e:
        raise gr.Error(f'Trim failed: {e}')


def ui_split_silence(audio_in, min_silence_ms, silence_thresh_dbfs):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        base = os.path.splitext(os.path.basename(src))[0]
        out_dir = f'/content/audio_out/{base}_chunks_by_silence'
        paths = split_by_silence(src, out_dir, int(min_silence_ms), int(silence_thresh_dbfs))
        if not paths:
            return None, None, 'No chunks found. Try lowering the silence threshold or shortening min_silence_len.'
        zip_path = f'/content/audio_out/{base}_chunks_by_silence.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for p in paths:
                zf.write(p, os.path.basename(p))
        # Show first chunk for preview
        s, sr = _load_audio(paths[0])
        preview_path = _gr_audio(s, sr)
        log = f'Split into {len(paths)} chunks (min silence {min_silence_ms}ms @ {silence_thresh_dbfs} dBFS)\nFirst chunk: {os.path.basename(paths[0])}'
        return preview_path, zip_path, log
    except Exception as e:
        raise gr.Error(f'Split failed: {e}')


def ui_split_chunks(audio_in, chunk_seconds, output_format):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        base = os.path.splitext(os.path.basename(src))[0]
        out_dir = f'/content/audio_out/{base}_chunks_{int(chunk_seconds)}s'
        paths = split_into_chunks(src, out_dir, float(chunk_seconds))
        if not paths:
            return None, None, 'Audio is shorter than the chunk size.'
        zip_path = f'/content/audio_out/{base}_chunks_{int(chunk_seconds)}s.zip'
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for p in paths:
                zf.write(p, os.path.basename(p))
        s, sr = _load_audio(paths[0])
        preview_path = _gr_audio(s, sr)
        log = f'Split into {len(paths)} chunks of {chunk_seconds}s each\nFirst chunk: {os.path.basename(paths[0])}'
        return preview_path, zip_path, log
    except Exception as e:
        raise gr.Error(f'Chunk split failed: {e}')


def ui_detect_silence(audio_in, min_silence_ms, silence_thresh_dbfs):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        ranges = detect_silence_ranges(src, int(min_silence_ms), int(silence_thresh_dbfs))
        if not ranges:
            return 'No silence ranges found with the current settings.'
        lines = [f'Found {len(ranges)} silence ranges (>= {min_silence_ms}ms @ {silence_thresh_dbfs} dBFS):']
        for i, (s, e) in enumerate(ranges[:50]):
            lines.append(f'  {i+1:3d}. {s:7.2f}s .. {e:7.2f}s   (duration: {e-s:.2f}s)')
        if len(ranges) > 50:
            lines.append(f'  ... and {len(ranges) - 50} more')
        return chr(10).join(lines)
    except Exception as e:
        raise gr.Error(f'Detect failed: {e}')


# ── Tab 3: Normalize ──────────────────────────────────────────
def ui_normalize(audio_in, mode, target_db, target_lufs, output_format, bitrate_kbps):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        samples, sr = _load_audio(src)
        loaded = _analyze(samples, sr)
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_normalized.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
        if mode == 'Peak':
            target_dbfs = float(target_db)
            _save_audio(samples, sr, out_path, format_ext=output_format, bitrate=br,
                        normalize_to_dbfs=target_dbfs)
            log = f'Peak normalized to {target_dbfs:.1f} dBFS'
        elif mode == 'LUFS (ITU-R BS.1770-4)':
            target = float(target_lufs)
            samples = _lufs_normalize(samples, sr, target)
            _save_audio(samples, sr, out_path, format_ext=output_format, bitrate=br,
                        normalize_to_dbfs=-1.0)
            log = f'LUFS normalized to {target:.1f} LUFS (ITU-R BS.1770-4 broadcast standard)'
        else:  # No-op copy
            _save_audio(samples, sr, out_path, format_ext=output_format, bitrate=br)
            log = 'No normalization (re-encoded only)'
        # We already have the final samples in memory (for Peak and No-op; LUFS too since we passed the modified samples to _save_audio).
        # For LUFS, we have the modified samples post-normalize. For Peak/No-op we have the original. Either way, no need to re-read the file.
        final_samples = samples
        final = _analyze(final_samples, sr)
        preview_path = _gr_audio(final_samples, sr)
        return preview_path, out_path, (
            f'Mode: {log}\n'
            f'Input:  peak={loaded["peak_dbfs"]:.1f} dBFS  RMS={loaded["rms_dbfs"]:.1f} dBFS  LUFS={loaded["lufs"]:.1f}\n'
            f'Output: peak={final["peak_dbfs"]:.1f} dBFS  RMS={final["rms_dbfs"]:.1f} dBFS  LUFS={final["lufs"]:.1f}\n'
            f'File: {out_path}'
        )
    except Exception as e:
        raise gr.Error(f'Normalize failed: {e}')


# ── Tab 5: Format Convert ─────────────────────────────────────
def ui_convert(audio_in, output_format, bitrate_kbps, target_sr_hz, mono_stereo):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        samples, sr = _load_audio(src)
        # Resample if requested
        if target_sr_hz and int(target_sr_hz) > 0 and int(target_sr_hz) != sr:
            target_sr = int(target_sr_hz)
            _, _, _, _, lr, _, _pb, _re = _ensure()
            samples = lr.resample(samples, orig_sr=sr, target_sr=target_sr).astype(np.float32)
            sr = target_sr
        # Mono/stereo
        if mono_stereo == 'Stereo':
            stereo = np.stack([samples, samples], axis=1)
            int_samples = (np.clip(stereo, -1, 1) * 32767).astype(np.int16)
            pd, sf, _, _, _, _, _pb, _re = _ensure()
            audio = pd.AudioSegment(
                data=int_samples.tobytes(),
                sample_width=2, frame_rate=sr, channels=2,
            )
            base = os.path.splitext(os.path.basename(src))[0]
            out_path = f'/content/audio_out/{base}_converted.{output_format}'
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            if output_format in ('wav', 'aiff'):
                sf.write(out_path, stereo, sr, subtype='PCM_16')
            else:
                codec, _ = CODEC_HINTS[output_format]
                audio.export(out_path, format=output_format,
                             parameters=['-acodec', codec, '-b:a', f'{int(bitrate_kbps)}k'])
        else:
            base = os.path.splitext(os.path.basename(src))[0]
            out_path = f'/content/audio_out/{base}_converted.{output_format}'
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
            _save_audio(samples, sr, out_path, format_ext=output_format, bitrate=br)
        log = f'Converted to {output_format} @ {sr} Hz, {mono_stereo.lower()}'
        # Load output ONCE for preview (Gradio audio element needs a file path that resolves to a known SR)
        out_samples, out_sr = _load_audio(out_path)
        preview_path = _gr_audio(out_samples, out_sr)
        return preview_path, out_path, log
    except Exception as e:
        raise gr.Error(f'Convert failed: {e}')


# ── Tab 7: Denoise ───────────────────────────────────────────
def ui_denoise(audio_in, backend, strength, run_dir, device_choice,
               output_format, bitrate_kbps, progress=gr.Progress()):
    """Denoise with one of two backends:
    - 'Spectral gate (noisereduce)': fast, lightweight, good for steady hum/hiss. Uses the strength slider.
    - 'Resemble denoiser (AI)': 44.1 kHz AI model, no bandwidth extension but better speech-isolating. Slow,
      resamples to 44.1 kHz mono. Uses run_dir + device from the AI Enhance source (shared).
    """
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        # Resolve run_dir: empty / placeholder -> None (use default)
        rd = (str(run_dir).strip() if run_dir else '')
        if rd in ('', 'None', '<default>'):
            rd = None
        # Resolve device: 'auto' -> None
        dev = (str(device_choice).strip().lower() if device_choice else 'auto')
        if dev in ('auto', 'none', ''):
            dev = None
        samples, sr = _load_audio(src)
        if backend == 'Resemble denoiser (AI, slow)':
            # Soft warning for very long files
            duration_min = len(samples) / sr / 60.0
            if duration_min > 10:
                print(f'[Denoise] WARNING: input is {duration_min:.1f} min long; Resemble denoiser is slow on long files. Consider trimming first.')
            progress(0.1, desc='Loading model...')
            result = res_denoise(samples, sr, run_dir=rd, device=dev)
            denoised = result['wav']; new_sr = result['sr']
            log = (f"Resemble denoiser (AI) applied.\n"
                   f"{result['log']}\n"
                   f"Note: output sample rate is 44.1 kHz (model native); input was {sr} Hz.\n"
                   f"Output: ?")
        else:
            denoised = _denoise(samples, sr, float(strength))
            new_sr = sr
            log = f'Spectral-gate denoise applied with strength {strength:.2f}'
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_denoised.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
        _save_audio(denoised, new_sr, out_path, format_ext=output_format, bitrate=br)
        log = log.replace('Output: ?', f'Output: {out_path}')
        preview_path = _gr_audio(denoised, new_sr)
        return preview_path, out_path, log
    except Exception as e:
        raise gr.Error(f'Denoise failed: {e}')


# ── AI Enhance (Resemble Enhance) ──────────────────────────────
def _ai_enhance_status() -> str:
    _, _, _, _, _, _, _, re = _ensure()
    if re is None:
        return ("Resemble Enhance is not installed. Run STEP 1 to install it.\n"
                "If install fails, the AI Enhance tab is disabled.")
    import torch as _t
    return f'Resemble Enhance ready. Device: {_re_device} | torch: {_t.__version__}'


def ui_ai_enhance(audio_in, solver, nfe, tau, lambd, denoise_first,
                  run_dir, device_choice,
                  output_format, bitrate_kbps, progress=gr.Progress()):
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        # Resolve run_dir: empty / placeholder -> None (use default)
        rd = (str(run_dir).strip() if run_dir else '')
        if rd in ('', 'None', '<default>'):
            rd = None
        # Resolve device: 'auto' -> None
        dev = (str(device_choice).strip().lower() if device_choice else 'auto')
        if dev in ('auto', 'none', ''):
            dev = None
        samples, sr = _load_audio(src)
        # Soft warning for very long files (AI Enhance is slow + memory-hungry on long audio)
        duration_min = len(samples) / sr / 60.0
        if duration_min > 10:
            print(f'[AI Enhance] WARNING: input is {duration_min:.1f} min long; AI Enhance is slow on long files. Consider trimming first.')
        progress(0.1, desc='Loading model...')
        result = res_enhance(samples, sr,
                             solver=str(solver), nfe=int(nfe),
                             tau=float(tau), lambd=float(lambd),
                             denoise_first=bool(denoise_first),
                             run_dir=rd, device=dev)
        progress(0.85, desc='Saving output...')
        out = result['wav']; new_sr = result['sr']
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_ai_enhanced.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
        _save_audio(out, new_sr, out_path, format_ext=output_format, bitrate=br)
        log = result['log'] + f'\nOutput: {out_path}'
        # Before/after waveforms
        before_path = _gr_audio(samples, sr)
        after_path = _gr_audio(out, new_sr)
        # Stats
        b_stats = _analyze(samples, sr)
        a_stats = _analyze(out, new_sr)
        stats = (f'BEFORE: dur {b_stats["duration_s"]:.2f}s @ {b_stats["sample_rate"]} Hz | '
                 f'peak {b_stats["peak_dbfs"]:.1f} dBFS | RMS {b_stats["rms_dbfs"]:.1f} dBFS | LUFS {b_stats["lufs"]:.1f}\n'
                 f'AFTER:  dur {a_stats["duration_s"]:.2f}s @ {a_stats["sample_rate"]} Hz | '
                 f'peak {a_stats["peak_dbfs"]:.1f} dBFS | RMS {a_stats["rms_dbfs"]:.1f} dBFS | LUFS {a_stats["lufs"]:.1f}\n'
                 f'Log:\n{result["log"]}')
        return (before_path, after_path,
                f'BEFORE: peak {b_stats["peak_dbfs"]:.1f} dBFS | RMS {b_stats["rms_dbfs"]:.1f} dBFS | LUFS {b_stats["lufs"]:.1f}',
                f'AFTER:  peak {a_stats["peak_dbfs"]:.1f} dBFS | RMS {a_stats["rms_dbfs"]:.1f} dBFS | LUFS {a_stats["lufs"]:.1f}',
                stats, out_path)
    except Exception as e:
        raise gr.Error(f'AI Enhance failed: {e}')


def ui_ai_enhance_batch(input_dir, pattern, recursive, output_dir,
                        solver, nfe, tau, lambd, denoise_first,
                        run_dir, device_choice,
                        output_format, bitrate_kbps, progress=gr.Progress()):
    if not input_dir or not os.path.isdir(input_dir):
        raise gr.Error('Provide a valid source directory.')
    if not output_dir:
        output_dir = os.path.join(input_dir, '_enhanced')
    try:
        rd = (str(run_dir).strip() if run_dir else '')
        if rd in ('', 'None', '<default>'):
            rd = None
        dev = (str(device_choice).strip().lower() if device_choice else 'auto')
        if dev in ('auto', 'none', ''):
            dev = None
        # Quick scan for very large files
        import glob as _glob
        scan = _glob.glob(os.path.join(input_dir, pattern or '*.wav'))
        for _p in scan[:5]:
            try:
                _s, _sr = _load_audio(_p)
                _dur = len(_s) / _sr / 60.0
                if _dur > 10:
                    print(f'[AI Enhance] WARNING: {os.path.basename(_p)} is {_dur:.1f} min long; AI Enhance is slow on long files.')
                    break
            except Exception:
                pass
        result = batch_res_enhance(
            input_dir=input_dir, output_dir=output_dir,
            pattern=pattern or '*.wav',
            recursive=bool(recursive),
            output_format=output_format, bitrate_kbps=int(bitrate_kbps),
            run_dir=rd, device=dev,
            solver=solver, nfe=int(nfe), tau=float(tau), lambd=float(lambd),
            denoise_first=bool(denoise_first),
            progress=progress,
        )
        return (result['log'],
                f'OK: {result["processed"]}  |  Failed: {result["failed"]}  |  Total: {result["total"]}  |  Time: {result["elapsed_s"]:.1f}s',
                result['output_dir'])
    except Exception as e:
        raise gr.Error(f'Batch AI Enhance failed: {e}')

# ── Tab 8: Batch ──────────────────────────────────────────────
def ui_batch(src_dir, preset_id, output_format, bitrate_kbps, progress=gr.Progress()):
    if not src_dir or not os.path.isdir(src_dir):
        raise gr.Error('Provide a directory of source audio files.')
    pipeline = get_preset(preset_id)
    pipeline['export_format'] = output_format
    exts = tuple('.' + f for f in IO_FORMATS)
    files = sorted(f for f in os.listdir(src_dir) if f.lower().endswith(exts))
    if not files:
        raise gr.Error(f'No supported audio files in {src_dir} (looked for {exts})')
    out_dir = '/content/audio_out/batch'
    os.makedirs(out_dir, exist_ok=True)
    br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else None
    zip_buf = io.BytesIO()
    log = []
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(files):
            src_path = os.path.join(src_dir, fname)
            base = os.path.splitext(fname)[0]
            out_path = os.path.join(out_dir, f'{base}_{preset_id}.{output_format}')
            try:
                result = process_audio(src_path, out_path, pipeline, bitrate=br)
                with open(out_path, 'rb') as f:
                    zf.writestr(os.path.basename(out_path), f.read())
                log.append(f'[{i+1:03d}/{len(files)}] ok · {result["final"]["duration_s"]:.1f}s · LUFS {result["loaded"]["lufs"]:.1f}->{result["final"]["lufs"]:.1f} · {fname}')
            except Exception as e:
                log.append(f'[{i+1:03d}/{len(files)}] FAILED · {e} · {fname}')
            progress((i+1)/max(len(files), 1), f'{i+1}/{len(files)}')
    zip_path = '/content/audio_out/batch.zip'
    with open(zip_path, 'wb') as f:
        f.write(zip_buf.getvalue())
    return zip_path, chr(10).join(log)


# ── Tab 9: Compare ───────────────────────────────────────────
def ui_compare(orig_in, processed_in, progress=gr.Progress()):
    if orig_in is None:
        raise gr.Error('Upload the ORIGINAL audio.')
    orig_src = _materialize_audio_input(orig_in)
    if not orig_src or not os.path.exists(orig_src):
        raise gr.Error('Invalid original audio.')
    try:
        o_samples, o_sr = _load_audio(orig_src)
        o_stats = _analyze(o_samples, o_sr)
    except Exception as e:
        raise gr.Error(f'Failed to load original: {e}')

    p_stats = None
    p_samples = None
    p_sr = None
    if processed_in is not None:
        proc_src = _materialize_audio_input(processed_in)
        try:
            p_samples, p_sr = _load_audio(proc_src)
            p_stats = _analyze(p_samples, p_sr)
        except Exception:
            p_stats = None
            p_samples = None
            p_sr = None

    def fmt(s, k):
        if s is None or s.get(k) is None: return '?'
        v = s[k]
        if isinstance(v, float) and not np.isfinite(v): return 'n/a'
        return v

    def pct(before, after):
        if before == 0 or after is None: return '-'
        try:
            return f'{(after - before):+.2f}'
        except Exception:
            return '-'

    rows = [
        ('Duration (s)',   f"{o_stats['duration_s']:.2f}",
                            f"{fmt(p_stats, 'duration_s') if p_stats else '?'}",
                            pct(o_stats['duration_s'], p_stats['duration_s'] if p_stats else None)),
        ('Sample rate',    f"{o_stats['sample_rate']:,}",
                            f"{fmt(p_stats, 'sample_rate') if p_stats else '?'}",
                            '-'),
        ('Peak (dBFS)',    f"{o_stats['peak_dbfs']:.1f}",
                            f"{fmt(p_stats, 'peak_dbfs') if p_stats else '?'}",
                            pct(o_stats['peak_dbfs'], p_stats['peak_dbfs'] if p_stats else None)),
        ('RMS (dBFS)',     f"{o_stats['rms_dbfs']:.1f}",
                            f"{fmt(p_stats, 'rms_dbfs') if p_stats else '?'}",
                            pct(o_stats['rms_dbfs'], p_stats['rms_dbfs'] if p_stats else None)),
        ('LUFS',           f"{o_stats['lufs']:.1f}",
                            f"{fmt(p_stats, 'lufs') if p_stats else '?'}",
                            pct(o_stats['lufs'], p_stats['lufs'] if p_stats else None)),
    ]
    log = chr(10).join(f'{r[0]:<15} {r[1]:>20}   {r[2]:>20}   {r[3]:>10}' for r in rows)
    # Build before/after waveform images
    before_png = _make_waveform_png(o_samples, o_sr, title='Original')
    after_png = _make_waveform_png(p_samples, p_sr, title='Processed') if p_samples is not None else None
    return (
        before_png,
        after_png,
        f'```\n{"Metric":<15} {"Original":>20}   {"Processed":>20}   {"Delta":>10}\n{"-"*70}\n{log}\n```',
    )


# ── Tab 4: Effects Chain (pedalboard) ────────────────────────
def ui_apply_effects_chain(audio_in, preset_id, output_format, bitrate_kbps,
                            hp_on, hp_hz, lp_on, lp_hz,
                            lad_on, lad_mode, lad_hz, lad_res,
                            gain_on, gain_db,
                            comp_on, comp_thr, comp_ratio, comp_atk, comp_rel,
                            lim_on, lim_thr, lim_rel,
                            rev_on, rev_room, rev_damp, rev_wet, rev_dry,
                            delay_on, delay_s, delay_fb, delay_mix,
                            chorus_on, chorus_rate, chorus_depth, chorus_mix, chorus_delay,
                            phaser_on, phaser_rate, phaser_depth, phaser_fb, phaser_mix, phaser_freq,
                            dist_on, dist_db, clip_on, clip_db,
                            pitch_on, pitch_semi,
                            bit_on, bit_depth, rsmp_on, rsmp_sr, mp3_on, mp3_q,
                            progress=gr.Progress()):
    """Apply a custom or preset effects chain. Builds a list of (effect_name, kwargs)
    from the enabled sliders, then runs through pedalboard.Pedalboard."""
    if audio_in is None:
        raise gr.Error('Upload an audio file first.')
    src = _materialize_audio_input(audio_in)
    if not src or not os.path.exists(src):
        raise gr.Error('Invalid audio input.')
    try:
        samples, sr = _load_audio(src)
        loaded = _analyze(samples, sr)
        base = os.path.splitext(os.path.basename(src))[0]
        out_path = f'/content/audio_out/{base}_fx.{output_format}'
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        # Build chain from preset OR from custom sliders
        log = [f'Input: {loaded["duration_s"]:.2f}s @ {loaded["sample_rate"]} Hz, peak {loaded["peak_dbfs"]:.1f} dBFS, LUFS {loaded["lufs"]:.1f}', '']
        if preset_id and preset_id != '_custom_' and preset_id in EFFECT_CHAIN_PRESETS:
            log.append(f'Chain preset: {EFFECT_CHAIN_PRESETS[preset_id]["name"]}')
            effected, apply_log = apply_chain_preset(preset_id, samples, sr)
            log.append(apply_log)
        else:
            chain = []
            # Build chain from sliders
            if hp_on:
                chain.append(('HighpassFilter', {'cutoff_hz': float(hp_hz)}))
            if lp_on:
                chain.append(('LowpassFilter', {'cutoff_hz': float(lp_hz)}))
            if lad_on:
                chain.append(('LadderFilter', {'mode': str(lad_mode), 'cutoff_hz': float(lad_hz), 'resonance': float(lad_res)}))
            if gain_on:
                chain.append(('Gain', {'gain_db': float(gain_db)}))
            if comp_on:
                chain.append(('Compressor', {'threshold_db': float(comp_thr), 'ratio': float(comp_ratio), 'attack_ms': float(comp_atk), 'release_ms': float(comp_rel)}))
            if rev_on:
                chain.append(('Reverb', {'room_size': float(rev_room), 'damping': float(rev_damp), 'wet_level': float(rev_wet), 'dry_level': float(rev_dry), 'width': 1.0}))
            if delay_on:
                chain.append(('Delay', {'delay_seconds': float(delay_s), 'feedback': float(delay_fb), 'mix': float(delay_mix)}))
            if chorus_on:
                chain.append(('Chorus', {'rate_hz': float(chorus_rate), 'depth': float(chorus_depth), 'centre_delay_ms': float(chorus_delay), 'feedback': 0.0, 'mix': float(chorus_mix)}))
            if phaser_on:
                chain.append(('Phaser', {'rate_hz': float(phaser_rate), 'depth': float(phaser_depth), 'centre_frequency_hz': float(phaser_freq), 'feedback': float(phaser_fb), 'mix': float(phaser_mix)}))
            if dist_on:
                chain.append(('Distortion', {'drive_db': float(dist_db)}))
            if clip_on:
                chain.append(('Clipping', {'threshold_db': float(clip_db)}))
            if pitch_on:
                chain.append(('PitchShift', {'semitones': float(pitch_semi)}))
            if bit_on:
                chain.append(('Bitcrush', {'bit_depth': float(bit_depth)}))
            if rsmp_on:
                chain.append(('Resample', {'target_sample_rate': float(rsmp_sr)}))
            if mp3_on:
                chain.append(('MP3Compressor', {'vbr_quality': float(mp3_q)}))
            # Always end with limiter if enabled
            if lim_on:
                chain.append(('Limiter', {'threshold_db': float(lim_thr), 'release_ms': float(lim_rel)}))
            log.append(f'Custom chain ({len(chain)} effects):')
            effected, apply_log = apply_effects_chain(samples, sr, chain)
            log.append(apply_log)
        log.append('')
        # Save
        br = f'{int(bitrate_kbps)}k' if output_format in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
        _save_audio(effected, sr, out_path, format_ext=output_format, bitrate=br)
        final = _analyze(effected, sr)
        log.append(f'Output: {final["duration_s"]:.2f}s, peak {final["peak_dbfs"]:.1f} dBFS, RMS {final["rms_dbfs"]:.1f} dBFS, LUFS {final["lufs"]:.1f}')
        log.append(f'File: {out_path} ({os.path.getsize(out_path)/1e3:.1f} KB)')
        preview_path = _gr_audio(effected, sr)
        return preview_path, out_path, chr(10).join(log)
    except Exception as e:
        raise gr.Error(f'Effects chain failed: {e}')


# ── Build the UI ────────────────────────────────────────────
with gr.Blocks(title='Audio Post-Processor', theme=gr.themes.Soft()) as demo:
    welcome_md = gr.Markdown(value='', visible=True)
    gr.Markdown('# Audio Post-Processor\n'
                'Trim · Split · Normalize · Effects · Convert · Denoise · Batch — for TTS / VC / podcast output.')

    with gr.Tabs():
        # ── Tab 1: Quick Process ──
        with gr.Tab('Quick Process'):
            gr.Markdown('**One-click preset processing** for the most common audio scenarios.')
            with gr.Row():
                with gr.Column():
                    q_audio = gr.Audio(label='Input audio (WAV, MP3, FLAC, OGG, OPUS, M4A, AAC, AIFF)',
                                        type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_voice_48k.wav'],
                                   ['/content/audio_examples/sample_noisy_48k.wav']],
                        inputs=[q_audio],
                        label='Try the example files',
                    )
                    q_preset = gr.Dropdown(
                        choices=[(p['name'], p['id']) for p in PRESETS],
                        value='podcast', label='Preset',
                        info='Podcast (default), Music, Speech, or Broadcast (EBU R128).',
                    )
                    q_format = gr.Dropdown(choices=IO_FORMATS, value='mp3', label='Output format',
                                            info='MP3 for sharing, WAV for editing, FLAC for archival.')
                    q_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps, for compressed formats)',
                                           info='Higher = better quality, bigger file. 192 is a good default.')
                    q_btn = gr.Button('Process', variant='primary')
                with gr.Column():
                    q_out = gr.Audio(label='Processed audio', type='filepath')
                    q_file = gr.File(label='Download')
                    q_log = gr.Textbox(label='Pipeline log', lines=12, interactive=False,
                                        info='One line per stage: load -> high-pass -> denoise -> LUFS -> peak -> export.')

            q_btn.click(ui_quick_process, [q_audio, q_preset, q_format, q_bitrate],
                        [q_out, q_file, q_log])

        # ── Tab 2: Trim & Split ──
        with gr.Tab('Trim & Split'):
            gr.Markdown('**Trim to a time range, split by silence, or split into N-second chunks.**')
            with gr.Row():
                with gr.Column():
                    t_audio = gr.Audio(label='Input audio', type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_voice_48k.wav']],
                        inputs=[t_audio], label='Try the example',
                    )
                    with gr.Accordion('Trim to time range', open=True):
                        trim_start = gr.Slider(0.0, 600.0, value=0.0, step=0.1, label='Start (s)',
                                                info='Keep audio starting at this time.')
                        trim_end = gr.Slider(0.0, 600.0, value=10.0, step=0.1, label='End (s)',
                                              info='Keep audio up to this time.')
                        trim_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                        trim_btn = gr.Button('Trim', variant='primary')
                with gr.Column():
                    trim_out = gr.Audio(label='Trimmed audio', type='filepath')
                    trim_file = gr.File(label='Download')
                    trim_log = gr.Textbox(label='Trim log', lines=3, interactive=False)

            trim_btn.click(ui_trim, [t_audio, trim_start, trim_end, trim_format],
                            [trim_out, trim_file, trim_log])

            gr.Markdown('---')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('**Split by silence** (uses pydub.silence.split_on_silence)')
                    s_min_ms = gr.Slider(100, 5000, value=500, step=100, label='Min silence length (ms)',
                                          info='A silence of at least this duration becomes a split point.')
                    s_thresh = gr.Slider(-60, -10, value=-40, step=1, label='Silence threshold (dBFS)',
                                          info='Audio quieter than this is considered silence. Lower = stricter.')
                    s_btn = gr.Button('Split by silence', variant='primary')
                with gr.Column():
                    s_out = gr.Audio(label='First chunk preview', type='filepath')
                    s_zip = gr.File(label='Download zip of all chunks')
                    s_log = gr.Textbox(label='Split log', lines=3, interactive=False)

            s_btn.click(ui_split_silence, [t_audio, s_min_ms, s_thresh],
                        [s_out, s_zip, s_log])

            gr.Markdown('---')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('**Split into N-second chunks** (equal-length)')
                    c_seconds = gr.Slider(1.0, 60.0, value=10.0, step=0.5, label='Chunk length (s)',
                                           info='Each output file will be exactly this long (last one may be shorter).')
                    c_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                    c_btn = gr.Button('Split into chunks', variant='primary')
                with gr.Column():
                    c_out = gr.Audio(label='First chunk preview', type='filepath')
                    c_zip = gr.File(label='Download zip of all chunks')
                    c_log = gr.Textbox(label='Chunk log', lines=3, interactive=False)

            c_btn.click(ui_split_chunks, [t_audio, c_seconds, c_format],
                        [c_out, c_zip, c_log])

            gr.Markdown('---')
            with gr.Row():
                with gr.Column():
                    gr.Markdown('**Detect silence ranges** (analysis only, no file output)')
                    d_min_ms = gr.Slider(100, 5000, value=500, step=100, label='Min silence length (ms)')
                    d_thresh = gr.Slider(-60, -10, value=-40, step=1, label='Silence threshold (dBFS)')
                    d_btn = gr.Button('Detect silences', variant='primary')
                with gr.Column():
                    d_log = gr.Textbox(label='Silence ranges (start..end seconds)', lines=12, interactive=False)

            d_btn.click(ui_detect_silence, [t_audio, d_min_ms, d_thresh], [d_log])

        # ── Tab 3: Normalize ──
        with gr.Tab('Normalize'):
            gr.Markdown('**Normalize loudness** for consistent playback. Three modes: peak, LUFS (ITU-R BS.1770-4), or no-op re-encode.')
            with gr.Row():
                with gr.Column():
                    n_audio = gr.Audio(label='Input audio', type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_voice_48k.wav']],
                        inputs=[n_audio], label='Try the example',
                    )
                    n_mode = gr.Radio(['Peak', 'LUFS (ITU-R BS.1770-4)', 'No-op (re-encode only)'],
                                       value='LUFS (ITU-R BS.1770-4)', label='Mode',
                                       info='LUFS is the broadcast standard. Peak is simpler and faster.')
                    n_target_db = gr.Slider(-24.0, -1.0, value=-1.0, step=0.5,
                                             label='Target peak (dBFS, when Peak mode)',
                                             info='-1 dBFS is a safe default that leaves headroom.')
                    n_target_lufs = gr.Slider(-30.0, -6.0, value=-16.0, step=0.5,
                                                label='Target LUFS (when LUFS mode)',
                                                info='-16 LUFS for podcast, -14 for Spotify, -23 for EBU R128 broadcast.')
                    n_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                    n_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                    n_btn = gr.Button('Normalize', variant='primary')
                with gr.Column():
                    n_out = gr.Audio(label='Normalized audio', type='filepath')
                    n_file = gr.File(label='Download')
                    n_log = gr.Textbox(label='Normalization report', lines=6, interactive=False,
                                        info='Shows before/after peak, RMS, LUFS.')

            n_btn.click(ui_normalize,
                        [n_audio, n_mode, n_target_db, n_target_lufs, n_format, n_bitrate],
                        [n_out, n_file, n_log])

        # ── Tab 4: Effects Chain (pedalboard) ──
        with gr.Tab('Effects Chain'):
            gr.Markdown('**Mastering & effects chain** powered by [Spotify `pedalboard`](https://github.com/spotify/pedalboard). Pick a preset, or build your own chain below. Effects run in order. All parameters live-update; the final limiter prevents clipping.')
            with gr.Row():
                with gr.Column():
                    ec_audio = gr.Audio(label='Input audio', type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_voice_48k.wav'],
                                   ['/content/audio_examples/sample_noisy_48k.wav']],
                        inputs=[ec_audio], label='Try the examples',
                    )
                    ec_preset = gr.Dropdown(
                        choices=[(p['name'], k) for k, p in EFFECT_CHAIN_PRESETS.items()] + [('(Custom - use sliders below)', '_custom_')],
                        value='podcast_voice', label='Chain preset',
                        info='Pre-built mastering chains. Pick one or choose Custom and tweak the sliders below.',
                    )
                    with gr.Accordion('Filters (Highpass / Lowpass / Ladder)', open=False):
                        ec_hp_on = gr.Checkbox(label='HighpassFilter', value=False,
                                                info='Cut rumble & mic handling noise below this frequency.')
                        ec_hp_hz = gr.Slider(20, 2000, value=80, step=10, label='Cutoff (Hz)',
                                              info='Try 80 for voice, 20 for music.')
                        ec_lp_on = gr.Checkbox(label='LowpassFilter', value=False,
                                                info='Cut high-frequency hiss above this frequency.')
                        ec_lp_hz = gr.Slider(1000, 20000, value=20000, step=100, label='Cutoff (Hz)',
                                              info='Try 8000 for lo-fi, 20000 (off) for clean.')
                        ec_lad_on = gr.Checkbox(label='LadderFilter (Moog-style)', value=False,
                                                 info='Vintage Moog ladder filter with selectable mode.')
                        with gr.Row():
                            ec_lad_mode = gr.Radio(['LPF12', 'LPF24', 'BPF12', 'HPF12'], value='LPF12', label='Mode',
                                                    info='LPF12 = 12 dB/oct lowpass (warm). HPF12 = 12 dB/oct highpass.')
                            ec_lad_hz = gr.Slider(20, 20000, value=2000, step=50, label='Cutoff (Hz)')
                            ec_lad_res = gr.Slider(0.0, 1.0, value=0.0, step=0.05, label='Resonance')
                    with gr.Accordion('Dynamics (Gain / Compressor / Limiter)', open=True):
                        ec_gain_on = gr.Checkbox(label='Gain (trim)', value=False,
                                                  info='Adjust overall level in dB.')
                        ec_gain_db = gr.Slider(-24, 24, value=0.0, step=0.5, label='Gain (dB)')
                        ec_comp_on = gr.Checkbox(label='Compressor', value=False,
                                                  info='Reduce dynamic range. Lower threshold + higher ratio = more compression.')
                        with gr.Row():
                            ec_comp_thr = gr.Slider(-60, 0, value=-18, step=1, label='Threshold (dB)')
                            ec_comp_ratio = gr.Slider(1.0, 20.0, value=4.0, step=0.5, label='Ratio')
                        with gr.Row():
                            ec_comp_atk = gr.Slider(0.1, 100, value=10, step=0.5, label='Attack (ms)')
                            ec_comp_rel = gr.Slider(10, 1000, value=100, step=10, label='Release (ms)')
                        ec_lim_on = gr.Checkbox(label='Limiter (always recommended)', value=True,
                                                 info='Brick-wall limiter. Prevents clipping. Keep on for safe output.')
                        with gr.Row():
                            ec_lim_thr = gr.Slider(-12, 0, value=-1.0, step=0.1, label='Threshold (dB)')
                            ec_lim_rel = gr.Slider(1, 1000, value=100, step=10, label='Release (ms)')
                    with gr.Accordion('Time & Space (Reverb / Delay / Chorus / Phaser)', open=False):
                        ec_rev_on = gr.Checkbox(label='Reverb', value=False,
                                                 info='Add room/space.')
                        with gr.Row():
                            ec_rev_room = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Room size')
                            ec_rev_damp = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Damping')
                        with gr.Row():
                            ec_rev_wet = gr.Slider(0.0, 1.0, value=0.33, step=0.05, label='Wet level')
                            ec_rev_dry = gr.Slider(0.0, 1.0, value=0.4, step=0.05, label='Dry level')
                        ec_delay_on = gr.Checkbox(label='Delay (echo)', value=False)
                        with gr.Row():
                            ec_delay_s = gr.Slider(0.01, 2.0, value=0.25, step=0.01, label='Delay (s)')
                            ec_delay_fb = gr.Slider(0.0, 0.95, value=0.3, step=0.05, label='Feedback')
                            ec_delay_mix = gr.Slider(0.0, 1.0, value=0.3, step=0.05, label='Mix')
                        ec_chorus_on = gr.Checkbox(label='Chorus', value=False)
                        with gr.Row():
                            ec_chorus_rate = gr.Slider(0.0, 10.0, value=1.0, step=0.1, label='Rate (Hz)')
                            ec_chorus_depth = gr.Slider(0.0, 1.0, value=0.25, step=0.05, label='Depth')
                            ec_chorus_mix = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Mix')
                            ec_chorus_delay = gr.Slider(1.0, 50.0, value=7.0, step=0.5, label='Centre delay (ms)',
                                                        info='Time offset between the two chorus voices. Higher = thicker / wider.')
                        ec_phaser_on = gr.Checkbox(label='Phaser', value=False)
                        with gr.Row():
                            ec_phaser_rate = gr.Slider(0.0, 10.0, value=0.5, step=0.1, label='Rate (Hz)')
                            ec_phaser_depth = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Depth')
                            ec_phaser_fb = gr.Slider(0.0, 0.95, value=0.0, step=0.05, label='Feedback')
                            ec_phaser_mix = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label='Mix')
                            ec_phaser_freq = gr.Slider(50.0, 5000.0, value=1300.0, step=50, label='Centre freq (Hz)',
                                                         info='Centre frequency of the all-pass filter sweep. Higher = brighter phaser.')
                    with gr.Accordion('Distortion & Pitch (Distortion / Clipping / PitchShift)', open=False):
                        ec_dist_on = gr.Checkbox(label='Distortion', value=False,
                                                  info='Soft saturation / overdrive.')
                        ec_dist_db = gr.Slider(0, 50, value=25, step=1, label='Drive (dB)')
                        ec_clip_on = gr.Checkbox(label='Clipping', value=False,
                                                  info='Hard clip at threshold (digital distortion).')
                        ec_clip_db = gr.Slider(-60, 0, value=-6, step=1, label='Threshold (dB)')
                        ec_pitch_on = gr.Checkbox(label='PitchShift', value=False,
                                                   info='Shift pitch in semitones (+/- 12 = one octave).')
                        ec_pitch_semi = gr.Slider(-24, 24, value=0, step=1, label='Semitones')
                    with gr.Accordion('Lo-Fi (Bitcrush / Resample / MP3Compressor)', open=False):
                        ec_bit_on = gr.Checkbox(label='Bitcrush', value=False,
                                                 info='Reduce bit depth for lo-fi digital sound.')
                        ec_bit_depth = gr.Slider(1, 16, value=8, step=0.5, label='Bit depth')
                        ec_rsmp_on = gr.Checkbox(label='Resample (sample-rate reduction)', value=False,
                                                  info='Downsample to simulate low-fi recording.')
                        ec_rsmp_sr = gr.Slider(1000, 44100, value=8000, step=500, label='Target SR (Hz)')
                        ec_mp3_on = gr.Checkbox(label='MP3 compression simulation', value=False,
                                                 info='Apply perceptual lossy compression artifacts.')
                        ec_mp3_q = gr.Slider(0.0, 10.0, value=2.0, step=0.5, label='VBR quality (0=best, 10=worst)')
                    ec_format = gr.Dropdown(choices=IO_FORMATS, value='mp3', label='Output format',
                                             info='WAV for editing, MP3 for sharing.')
                    ec_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                    with gr.Row():
                        ec_btn = gr.Button('Apply effects chain', variant='primary')
                        ec_clear = gr.ClearButton(
                            components=[ec_hp_on, ec_lp_on, ec_lad_on, ec_gain_on, ec_comp_on, ec_lim_on,
                                       ec_rev_on, ec_delay_on, ec_chorus_on, ec_phaser_on,
                                       ec_dist_on, ec_clip_on, ec_pitch_on, ec_bit_on, ec_rsmp_on, ec_mp3_on],
                            value='Uncheck all custom effects',
                            label='Uncheck all custom effects',
                        )
                with gr.Column():
                    ec_out = gr.Audio(label='Processed audio', type='filepath')
                    ec_file = gr.File(label='Download')
                    ec_log = gr.Textbox(label='Effects chain log', lines=15, interactive=False,
                                         info='Each enabled effect and its parameters, then the final limiter.')
            ec_btn.click(ui_apply_effects_chain,
                          [ec_audio, ec_preset, ec_format, ec_bitrate,
                           ec_hp_on, ec_hp_hz, ec_lp_on, ec_lp_hz,
                           ec_lad_on, ec_lad_mode, ec_lad_hz, ec_lad_res,
                           ec_gain_on, ec_gain_db,
                           ec_comp_on, ec_comp_thr, ec_comp_ratio, ec_comp_atk, ec_comp_rel,
                           ec_lim_on, ec_lim_thr, ec_lim_rel,
                           ec_rev_on, ec_rev_room, ec_rev_damp, ec_rev_wet, ec_rev_dry,
                           ec_delay_on, ec_delay_s, ec_delay_fb, ec_delay_mix,
                           ec_chorus_on, ec_chorus_rate, ec_chorus_depth, ec_chorus_mix, ec_chorus_delay,
                           ec_phaser_on, ec_phaser_rate, ec_phaser_depth, ec_phaser_fb, ec_phaser_mix, ec_phaser_freq,
                           ec_dist_on, ec_dist_db, ec_clip_on, ec_clip_db,
                           ec_pitch_on, ec_pitch_semi,
                           ec_bit_on, ec_bit_depth, ec_rsmp_on, ec_rsmp_sr, ec_mp3_on, ec_mp3_q],
                          [ec_out, ec_file, ec_log])

        # ── Tab 5: Format Convert ──
        with gr.Tab('Format Convert'):
            gr.Markdown('**Convert between 8 audio formats**. Pick bitrate (for lossy) and sample rate (for resample).')
            with gr.Row():
                with gr.Column():
                    cv_audio = gr.Audio(label='Input audio', type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_voice_48k.wav']],
                        inputs=[cv_audio], label='Try the example',
                    )
                    cv_format = gr.Dropdown(choices=IO_FORMATS, value='mp3', label='Output format',
                                             info='WAV/FLAC/AIFF = lossless. MP3/OGG/Opus/M4A/AAC = lossy.')
                    cv_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                    cv_sr = gr.Dropdown(
                        choices=['Keep original', '8000', '16000', '22050', '24000', '32000', '44100', '48000', '96000'],
                        value='Keep original', label='Sample rate (Hz)',
                        info='Phone = 8 kHz, TTS = 16-24 kHz, CD = 44.1 kHz, DVD/Blu-ray = 48 kHz.')
                    cv_channels = gr.Radio(['Mono', 'Stereo'], value='Mono', label='Channels',
                                            info='Mono for speech/TTS, stereo for music.')
                    cv_btn = gr.Button('Convert', variant='primary')
                with gr.Column():
                    cv_out = gr.Audio(label='Converted audio', type='filepath')
                    cv_file = gr.File(label='Download')
                    cv_log = gr.Textbox(label='Convert log', lines=4, interactive=False)

            cv_btn.click(ui_convert,
                         [cv_audio, cv_format, cv_bitrate, cv_sr, cv_channels],
                         [cv_out, cv_file, cv_log])

        
        
        with gr.Tab('AI Enhance'):
            gr.Markdown('**AI speech enhancement** via [Resemble Enhance](https://github.com/resemble-ai/resemble-enhance) (Resemble AI, MIT, 44.1 kHz). Combines a **denoiser** (separates voice from background) with an **enhancer** (restores high-frequency content lost in TTS / low-quality recordings, reduces artifacts). Works on noisy speech, thin TTS output, and compressed dialog.\n\n**Tips:**\n- First run downloads ~500 MB of model weights; subsequent runs are fast.\n- **GPU recommended** but **CPU supported** (10-60s per minute of audio on CPU, 1-5s on GPU).\n- **lambd** controls denoising strength inside the enhancer (0 = clean studio, 1 = aggressive).\n- **tau** controls the prior temperature (lower = more conservative, higher = more expressive).\n- **denoise_first** = True runs the standalone denoiser first, then enhances (slowest, highest quality).\n- For music or non-speech, use the **Denoise** tab instead (spectral gate is gentler).')
            ae_status = gr.Markdown(value=_ai_enhance_status(), label='Model status')
            with gr.Accordion('Model source & cache (optional)', open=False):
                ae_run_dir = gr.Textbox(value='<default>', label='Model run_dir (checkpoint folder)',
                                         info='Leave "<default>" to use the upstream ResembleAI checkpoint (auto-downloaded, ~713 MB). Set to a folder containing hparams.yaml + ds/G/default/mp_rank_00_model_states.pt to use a custom fine-tune.')
                ae_device = gr.Dropdown(choices=['auto', 'cuda', 'cpu'], value='auto', label='Device',
                                         info='Compute device. auto = use GPU if available, else CPU. cuda = force GPU (errors if unavailable). cpu = force CPU (slower but consistent).')
            with gr.Tabs():
                with gr.Tab('Single file'):
                    with gr.Row():
                        with gr.Column():
                            ae_audio = gr.Audio(label='Input audio', type='filepath',
                                                info='Speech or dialog. Resamples internally to 44.1 kHz mono.')
                            gr.Examples(
                                examples=[['/content/audio_examples/sample_voice_48k.wav'],
                                          ['/content/audio_examples/sample_noisy_48k.wav']],
                                inputs=[ae_audio], label='Try the demo samples',
                            )
                            with gr.Accordion('Model hyperparameters', open=True):
                                ae_solver = gr.Dropdown(choices=['Midpoint', 'RK4', 'Euler'],
                                                         value='Midpoint', label='CFM ODE Solver',
                                                         info='Numerical solver. Midpoint is the default; RK4 is more accurate but ~2x slower; Euler is fastest.')
                                ae_nfe = gr.Slider(1, 128, value=64, step=1,
                                                    label='CFM NFE (function evaluations)',
                                                    info='Higher = better quality, slower. 32 = fast, 64 = default, 128 = max quality.')
                                ae_tau = gr.Slider(0.0, 1.0, value=0.5, step=0.01,
                                                    label='CFM prior temperature (tau)',
                                                    info='Lower = closer to training distribution (safer). Higher = more variation.')
                                ae_lambd = gr.Slider(0.0, 1.0, value=0.5, step=0.01,
                                                      label='Denoiser strength (lambd)',
                                                      info='0 = no denoise, 1 = aggressive. 0.5 is a good default for noisy speech.')
                                ae_denoise_first = gr.Checkbox(value=False, label='Run standalone denoiser first',
                                                                info='Two-pass: denoise then enhance. ~2x slower but cleaner for very noisy input.')
                            ae_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                            ae_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                            ae_btn = gr.Button('Enhance', variant='primary')
                        with gr.Column():
                            ae_before = gr.Audio(label='BEFORE (original)', type='filepath')
                            ae_after = gr.Audio(label='AFTER (enhanced)', type='filepath')
                            ae_stats_before = gr.Textbox(label='BEFORE stats', lines=1, interactive=False)
                            ae_stats_after = gr.Textbox(label='AFTER stats', lines=1, interactive=False)
                            ae_log = gr.Textbox(label='Log + LUFS / RMS comparison', lines=8, interactive=False)
                            ae_file = gr.File(label='Download enhanced audio')

                    ae_btn.click(ui_ai_enhance,
                                 [ae_audio, ae_solver, ae_nfe, ae_tau, ae_lambd, ae_denoise_first,
                                  ae_run_dir, ae_device,
                                  ae_format, ae_bitrate],
                                 [ae_before, ae_after, ae_stats_before, ae_stats_after, ae_log, ae_file])
                    # Refresh model status after single-file run completes
                    ae_btn.click(_ai_enhance_status, inputs=None, outputs=[ae_status])
                with gr.Tab('Batch (directory)'):
                    gr.Markdown('Enhance **every audio file in a directory**. Per-file failures are caught and logged; one bad file will not abort the batch. Output is saved as `<name>_enhanced.<format>` next to each input (or in a parallel subdirectory tree if recursive).')
                    with gr.Row():
                        with gr.Column():
                            bae_input_dir = gr.Textbox(label='Input directory',
                                                        placeholder='/content/audio_in/',
                                                        info='Folder containing audio files to enhance.')
                            bae_output_dir = gr.Textbox(label='Output directory (optional)',
                                                          placeholder='Defaults to <input_dir>/_enhanced/',
                                                          info='Where to write enhanced files. Created if missing.')
                            bae_pattern = gr.Dropdown(
                                choices=['*.wav', '*.flac', '*.mp3', '*.ogg', '*.opus', '*.m4a', '*.aac', '*.aiff', '*'],
                                value='*.wav', label='File pattern',
                                info='Glob pattern (lowercase). "*" matches all files.')
                            bae_recursive = gr.Checkbox(value=False, label='Recursive (include subdirectories)',
                                                         info='Walk subdirectories and preserve the relative folder structure in the output.')
                            with gr.Accordion('Model hyperparameters (shared by all files)', open=False):
                                bae_solver = gr.Dropdown(choices=['Midpoint', 'RK4', 'Euler'],
                                                          value='Midpoint', label='CFM ODE Solver',
                                                          info='Same as single-file mode.')
                                bae_nfe = gr.Slider(1, 128, value=64, step=1,
                                                     label='CFM NFE',
                                                     info='Higher = better, slower.')
                                bae_tau = gr.Slider(0.0, 1.0, value=0.5, step=0.01,
                                                     label='CFM prior temperature (tau)',
                                                     info='Lower = closer to training distribution.')
                                bae_lambd = gr.Slider(0.0, 1.0, value=0.5, step=0.01,
                                                       label='Denoiser strength (lambd)',
                                                       info='0 = no denoise, 1 = aggressive.')
                                bae_denoise_first = gr.Checkbox(value=False, label='Run standalone denoiser first',
                                                                 info='Two-pass: denoise then enhance.')
                            bae_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                            bae_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                            bae_btn = gr.Button('Enhance all files in directory', variant='primary')
                        with gr.Column():
                            bae_summary = gr.Textbox(label='Summary (OK / Failed / Total / Time)', lines=1, interactive=False)
                            bae_out_dir = gr.Textbox(label='Output directory', lines=1, interactive=False)
                            bae_log = gr.Textbox(label='Batch log (per-file results)', lines=15, interactive=False,
                                                  info='Shows per-file success/failure and a final summary.')

                    bae_btn.click(ui_ai_enhance_batch,
                                  [bae_input_dir, bae_pattern, bae_recursive, bae_output_dir,
                                   bae_solver, bae_nfe, bae_tau, bae_lambd, bae_denoise_first,
                                   ae_run_dir, ae_device,
                                   bae_format, bae_bitrate],
                                  [bae_log, bae_summary, bae_out_dir])
                    # Refresh model status after batch run completes
                    bae_btn.click(_ai_enhance_status, inputs=None, outputs=[ae_status])

        # ── Tab 7: Denoise ──
        with gr.Tab('Denoise'):
            gr.Markdown('**Noise reduction** for speech / dialog. Two backends: **Spectral gate** (fast, no model, `noisereduce`) and **Resemble denoiser** (44.1 kHz AI model — same engine as the AI Enhance tab, denoise-only path, no bandwidth extension). The Resemble backend resamples to 44.1 kHz mono internally.')
            dn_status = gr.Markdown(value=_ai_enhance_status(), label='Resemble model status')
            with gr.Row():
                with gr.Column():
                    dn_audio = gr.Audio(label='Input audio', type='filepath')
                    gr.Examples(
                        examples=[['/content/audio_examples/sample_noisy_48k.wav']],
                        inputs=[dn_audio], label='Try the noisy example',
                    )
                    dn_backend = gr.Radio(
                        choices=['Spectral gate (noisereduce)', 'Resemble denoiser (AI, slow)'],
                        value='Spectral gate (noisereduce)', label='Backend',
                        info='Spectral gate = fast, no model, good for steady hum/hiss. Resemble denoiser = 44.1 kHz AI, better for speech but slow (1-5s/min on GPU, 10-60s on CPU).'
                    )
                    dn_strength = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label='Strength (spectral gate only)',
                                             info='0 = no denoise, 1 = aggressive. 0.5-0.7 is a good default. Higher = more risk of artifacts. Ignored when Resemble backend is selected.',
                                             visible=True)
                    with gr.Group(visible=False) as dn_re_group:
                        gr.Markdown('**Resemble denoiser options** (shared with the AI Enhance tab; leave defaults unless you have a custom checkpoint).')
                        dn_run_dir = gr.Textbox(value='<default>', label='Model run_dir (checkpoint folder)',
                                                 info='Leave "<default>" to use the upstream ResembleAI checkpoint (auto-downloaded, ~713 MB). Set to a folder containing hparams.yaml + ds/G/default/mp_rank_00_model_states.pt to use a custom fine-tune.')
                        dn_device = gr.Dropdown(choices=['auto', 'cuda', 'cpu'], value='auto', label='Device',
                                                 info='Compute device. auto = use GPU if available, else CPU.')
                    dn_format = gr.Dropdown(choices=IO_FORMATS, value='wav', label='Output format')
                    dn_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                    dn_btn = gr.Button('Denoise', variant='primary')
                with gr.Column():
                    dn_out = gr.Audio(label='Denoised audio', type='filepath')
                    dn_file = gr.File(label='Download')
                    dn_log = gr.Textbox(label='Denoise log', lines=6, interactive=False)

            # Show/hide Resemble-specific options based on backend choice
            def _toggle_re(backend):
                return gr.Group(visible=backend.startswith('Resemble'))
            dn_backend.change(_toggle_re, inputs=[dn_backend], outputs=[dn_re_group])
            # Refresh model status after a run
            dn_btn.click(ui_denoise, [dn_audio, dn_backend, dn_strength, dn_run_dir, dn_device,
                                       dn_format, dn_bitrate],
                         [dn_out, dn_file, dn_log])
            dn_btn.click(_ai_enhance_status, inputs=None, outputs=[dn_status])

        # ── Tab 8: Batch ──
        with gr.Tab('Batch'):
            gr.Markdown('**Apply any preset to every audio file in a directory.** Outputs a zip of all processed files.')
            with gr.Row():
                with gr.Column():
                    b_dir = gr.Textbox(label='Source directory', value='/content/audio_out/batch_in',
                                        info='Path to a directory of audio files. Supported: .wav, .mp3, .flac, .ogg, .opus, .m4a, .aac, .aiff')
                    b_preset = gr.Dropdown(choices=[(p['name'], p['id']) for p in PRESETS], value='podcast',
                                            label='Preset')
                    b_format = gr.Dropdown(choices=IO_FORMATS, value='mp3', label='Output format')
                    b_bitrate = gr.Slider(32, 320, value=192, step=32, label='Bitrate (kbps)')
                    b_btn = gr.Button('Run batch', variant='primary')
                with gr.Column():
                    b_zip = gr.File(label='Download .zip of processed files')
                    b_log = gr.Textbox(label='Per-file log', lines=10, interactive=False)

            b_btn.click(ui_batch, [b_dir, b_preset, b_format, b_bitrate], [b_zip, b_log])

        # ── Tab 9: Compare ──
        with gr.Tab('Compare'):
            gr.Markdown('**Side-by-side comparison** of original vs processed. Shows waveforms + stats (peak, RMS, LUFS, duration).')
            with gr.Row():
                with gr.Column():
                    cmp_orig = gr.Audio(label='Original audio', type='filepath')
                with gr.Column():
                    cmp_proc = gr.Audio(label='Processed audio', type='filepath')
            cmp_btn = gr.Button('Compare', variant='primary')
            with gr.Row():
                with gr.Column():
                    cmp_wave_orig = gr.Image(label='Original waveform', type='pil')
                with gr.Column():
                    cmp_wave_proc = gr.Image(label='Processed waveform', type='pil')
            cmp_log = gr.Markdown(label='Side-by-side stats')

            cmp_btn.click(ui_compare, [cmp_orig, cmp_proc], [cmp_wave_orig, cmp_wave_proc, cmp_log])

        # ── Tab 10: Help ──
        with gr.Tab('Help'):
            gr.Markdown(
                """## When to use each preset

| Preset | Best for | Output LUFS | Peak | Denoise |
| --- | --- | --- | --- | --- |
| **Podcast** | Two-speaker voice | -16 LUFS | -1 dBFS | 0.5 |
| **Music** | Music (preserves bass) | -14 LUFS | -1 dBFS | 0.3 |
| **Speech** | Broadcast-ready voice | -23 LUFS | -2 dBFS | 0.7 |
| **Broadcast** | EBU R128 / ATSC A/85 | -23 LUFS | -1 dBFS | 0.4 |
| **Studio Polish** *(AI)* | Thin / compressed TTS or noisy speech | -16 LUFS | -1 dBFS | Resemble Enhance (44.1 kHz AI) |

## When to use each tab

- **Quick Process** -- one-click preset (90% of use cases)
- **Trim & Split** -- remove silence at start/end, or split a long file into chunks for further processing
- **Normalize** -- re-loudness an already-processed file to a specific LUFS target
- **Effects Chain** -- studio mastering with `pedalboard` (compressor, limiter, EQ, reverb, delay, chorus, phaser, distortion, pitch shift, bitcrush, etc.). 4 chain presets (Podcast Voice, Vocal Cleaning, Music Mastering, Lo-Fi Tape) or build a custom chain.
- **Format Convert** -- change container/codec (WAV to MP3, MP3 to Opus, etc.) with optional resample
- **AI Enhance** -- Resemble Enhance (44.1 kHz AI model, Resemble AI MIT): denoise + bandwidth extension + artifact reduction. Best for thin / compressed TTS output or noisy speech. Slow (1-5s/min on GPU, 10-60s on CPU). Single file or batch directory mode.
- **Denoise** -- noise reduction with two backends: **Spectral gate** (`noisereduce`, fast, no model, good for steady hum/hiss) and **Resemble denoiser** (44.1 kHz AI, slow but better for speech, shares settings with AI Enhance tab)
- **Batch** -- apply any preset to a directory of files at once
- **Compare** -- A/B test the same audio before and after processing

## Format cheatsheet

| Format | Lossy? | Best for | License |
| --- | --- | --- | --- |
| **WAV** | No (PCM) | Editing, archival | n/a (container) |
| **FLAC** | No (lossless compressed) | Archival, ~60% of WAV size | BSD |
| **AIFF** | No (PCM) | macOS, Logic Pro | n/a (container) |
| **MP3** | Yes | Universal sharing | Patent-expired |
| **OGG Vorbis** | Yes | Open-source alternative to MP3 | BSD |
| **Opus** | Yes | Streaming, voice, low bitrate | BSD |
| **M4A / AAC** | Yes | Apple ecosystem, YouTube | Via MPEG LA |
| **AAC ADTS** | Yes | Raw stream (no container) | Via MPEG LA |

## Loudness targets cheat-sheet

| Target LUFS | Use case |
| --- | --- |
| -14 LUFS | Spotify, YouTube, Apple Music (music default) |
| -16 LUFS | Podcast, audiobook default |
| -18 LUFS | AES streaming recommendation |
| -23 LUFS | EBU R128 / ATSC A/85 broadcast standard |
| -24 LUFS | ATSC A/85 (digital TV, US) |
| -27 LUFS | Cinematic / film dialog reference |

## Tips for AI Enhance

- **First run downloads ~713 MB** of model weights to `HF_HOME` (set to Drive in Step 1 for caching across Colab sessions)
- **GPU strongly recommended.** CPU works but is 10-60x slower. Use the **Device** dropdown to force CPU for consistent benchmarking.
- **NFE slider**: 32 = fast preview, 64 = default, 128 = max quality (~2x slower than 32)
- **lambd slider**: 0 = no denoise inside enhancer, 1 = aggressive. Default 0.5 works for most speech.
- **denoise_first checkbox**: 2-pass (denoise then enhance). Slowest but cleanest for very noisy input.
- **For music or non-speech**, use the **Denoise** tab instead (spectral gate is gentler, doesn't try to "restore" missing content).
- **Custom fine-tunes**: point the **Model run_dir** field at a folder containing `hparams.yaml` + `ds/G/default/mp_rank_00_model_states.pt`.

## Citation

This notebook wraps:
- [jiaaro/pydub](https://github.com/jiaaro/pydub) -- MIT
- [imageio/imageio-ffmpeg](https://github.com/imageio/imageio-ffmpeg) -- BSD-4-Clause
- [bastibe/python-soundfile](https://github.com/bastibe/python-soundfile) -- BSD-3-Clause (libsndfile)
- [librosa/librosa](https://github.com/librosa/librosa) -- ISC
- [csteinmetz1/pyloudnorm](https://github.com/csteinmetz1/pyloudnorm) -- MIT
- [timsainb/noisereduce](https://github.com/timsainb/noisereduce) -- MIT
- [spotify/pedalboard](https://github.com/spotify/pedalboard) -- GPL-3.0 (used as a `pip install` dependency; notebook remains MIT)
- [resemble-ai/resemble-enhance](https://github.com/resemble-ai/resemble-enhance) -- MIT (Resemble AI)
"""
            )

    # Welcome message (populated into welcome_md on page load)
    def _welcome():
        return (
            '> **Welcome to Audio Post-Processor** — the companion to all our TTS / VC / voice-cloning notebooks.\n\n'
            '> **Quick Start:** Pick the **Quick Process** tab → upload an audio file (WAV, MP3, FLAC, OGG, OPUS, M4A, AAC, or AIFF) → '
            'choose a preset (Podcast is the most common) → pick an output format (MP3 for sharing, WAV for editing) → click **Process**.\n\n'
            '> **Other tabs:** **Trim & Split** for silence detect / chunking, **Normalize** for LUFS / peak re-loudness, **Effects Chain** for studio mastering with `pedalboard`, '
            '**Format Convert** for codec / sample-rate changes, **AI Enhance** for AI denoise + bandwidth extension ([Resemble Enhance](https://github.com/resemble-ai/resemble-enhance), MIT, slow but powerful), '
            '**Denoise** for noise reduction (spectral-gate fast, or Resemble AI slow but better for speech), **Batch** to apply any preset to a directory, **Compare** to A/B test, **Help** for cheatsheets. '
            'See the **Help** tab for the full cheatsheet (presets, formats, LUFS targets, citations).'
        )

    demo.load(_welcome, inputs=None, outputs=[welcome_md])


# ── Launch ────────────────────────────────────────────────
demo.queue(max_size=8, concurrency_limit=2)
demo.launch(share=False, show_error=True, height=900, quiet=False)



In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')



In [ ]:
# @title Step 6 — Quick Test (one-off processing, no UI)
# @markdown Run a single preset from the notebook. Useful for scripting or quick checks.

PRESET = "podcast"  # @param ["podcast", "music", "speech", "broadcast"]
INPUT = ""  # @param {type:"string"}
OUTPUT_FORMAT = "mp3"  # @param ["wav", "mp3", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
BITRATE_KBPS = 192  # @param {type:"slider", min:32, max:320, step:32}

import os, time
if not INPUT or not os.path.exists(INPUT):
    raise SystemExit('INPUT is required (path to a .wav/.mp3/.flac/.ogg/.opus/.m4a/.aac/.aiff file).')

pipeline = get_preset(PRESET)
pipeline['export_format'] = OUTPUT_FORMAT
base = os.path.splitext(os.path.basename(INPUT))[0]
out_path = f'/content/audio_out/{base}_{PRESET}.{OUTPUT_FORMAT}'
os.makedirs(os.path.dirname(out_path), exist_ok=True)

br = f'{int(BITRATE_KBPS)}k' if OUTPUT_FORMAT in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'

print(f'[Test] preset={PRESET} input={os.path.basename(INPUT)} output={OUTPUT_FORMAT} bitrate={br}')
t0 = time.time()
result = process_audio(INPUT, out_path, pipeline)
if OUTPUT_FORMAT in ('mp3', 'ogg', 'opus', 'm4a', 'aac'):
    _save_audio(result['samples'], result['sample_rate'], out_path,
                format_ext=OUTPUT_FORMAT, bitrate=br,
                normalize_to_dbfs=pipeline.get('peak_db', -1.0))
elapsed = time.time() - t0

print(f'\n[Done] {out_path}')
print(f'[Done] Total: {elapsed:.1f}s')
print(f'[Done] Input:  peak={result["loaded"]["peak_dbfs"]:.1f} dBFS  RMS={result["loaded"]["rms_dbfs"]:.1f} dBFS  LUFS={result["loaded"]["lufs"]:.1f}')
print(f'[Done] Output: peak={result["final"]["peak_dbfs"]:.1f} dBFS  RMS={result["final"]["rms_dbfs"]:.1f} dBFS  LUFS={result["final"]["lufs"]:.1f}')

from IPython.display import FileLink, display
display(FileLink(out_path))



In [ ]:
# @title Step 7 — Batch Processing
# @markdown Apply a preset to every audio file in a directory. Each input becomes one output.

PRESET = "podcast"  # @param ["podcast", "music", "speech", "broadcast"]
SRC_DIR = "/content/audio_out/batch_in"  # @param {type:"string"}
OUT_DIR = "/content/audio_out/batch"  # @param {type:"string"}
OUTPUT_FORMAT = "mp3"  # @param ["wav", "mp3", "flac", "ogg", "opus", "m4a", "aac", "aiff"]
BITRATE_KBPS = 192  # @param {type:"slider", min:32, max:320, step:32}

import os, time
if not SRC_DIR or not os.path.isdir(SRC_DIR):
    raise SystemExit(f'SRC_DIR is required and must be a directory (got {SRC_DIR!r}).')

pipeline = get_preset(PRESET)
pipeline['export_format'] = OUTPUT_FORMAT
os.makedirs(OUT_DIR, exist_ok=True)

exts = tuple('.' + f for f in IO_FORMATS)
files = sorted(f for f in os.listdir(SRC_DIR) if f.lower().endswith(exts))
if not files:
    raise SystemExit(f'No supported audio files in {SRC_DIR} (looked for {exts})')

br = f'{int(BITRATE_KBPS)}k' if OUTPUT_FORMAT in ('mp3', 'ogg', 'opus', 'm4a', 'aac') else '192k'
print(f'[Batch] {len(files)} files · preset={PRESET} · output={OUTPUT_FORMAT} · bitrate={br}')

t_start = time.time()
for i, fname in enumerate(files):
    src_path = os.path.join(SRC_DIR, fname)
    base = os.path.splitext(fname)[0]
    out_path = os.path.join(OUT_DIR, f'{base}_{PRESET}.{OUTPUT_FORMAT}')
    label = f"[{i+1:03d}/{len(files)}]"
    print(f'{label} {fname}', flush=True)
    t0 = time.time()
    try:
        result = process_audio(src_path, out_path, pipeline)
        if OUTPUT_FORMAT in ('mp3', 'ogg', 'opus', 'm4a', 'aac'):
            _save_audio(result['samples'], result['sample_rate'], out_path,
                        format_ext=OUTPUT_FORMAT, bitrate=br,
                        normalize_to_dbfs=pipeline.get('peak_db', -1.0))
        print(f'  ✓ {result["final"]["duration_s"]:.1f}s · LUFS {result["loaded"]["lufs"]:.1f}->{result["final"]["lufs"]:.1f} · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\n[Done] {len(files)} files in {OUT_DIR} · total wall time {elapsed:.1f}s')

